# HuggingFace 커스텀 서비스 프레임워크 최종 노트북
## NSMC → 카페 리뷰 운영 신호 → Store DNA → 서비스화

이 노트북의 목적은 단순히 NSMC 감성분석 모델을 한 번 돌려보는 것이 아닙니다.

**NSMC로 HuggingFace 파인튜닝의 기본 구조를 실제 학습 기준으로 검증한 뒤, 같은 구조를 카페 리뷰 운영 신호 분석으로 확장하고, 최종적으로 Store DNA와 서비스화 구조로 연결하는 것**이 핵심입니다.

즉, 흐름은 다음과 같습니다.

1. **NSMC 감성분석**: HuggingFace `Tokenizer → Dataset → Trainer → 평가` 흐름을 3 epoch 기준으로 검증
2. **카페 리뷰 운영 신호**: 단순 긍정/부정을 넘어 운영 개선 신호를 다중 라벨로 추출
3. **Store DNA**: 리뷰·매출·운영 데이터를 매장별 특징 프로파일로 누적
4. **서비스화**: 사장님이 이해할 수 있는 진단, 리포트, 추천 액션으로 변환

---

### 실행 기준

이 버전은 이전의 단순 실행 확인용 `1 epoch / smoke test` 버전이 아닙니다.
기본값은 다음과 같이 **실제 평가용 학습 설정**으로 잡았습니다.

- `NUM_EPOCHS = 3`
- `SMOKE_TEST = False`
- 전체 NSMC train/test split 사용
- STEP 4 기본 fine-tuning과 STEP 5 bucketing 실험 비교

단, 실행 환경에서 시간이 부족한 경우에만 초기화 셀에서 `SMOKE_TEST = True`로 바꿔 빠른 구조 확인용으로 실행할 수 있습니다.


# Part 1. NSMC 학습 · 파인튜닝 · 평가

이 파트는 HuggingFace 기반 커스텀 서비스 프레임워크의 **선행 검증 구간**입니다.

NSMC는 영화 리뷰 감성분석 데이터셋이지만, 여기서 중요한 것은 데이터셋 자체가 아니라 다음 구조입니다.

- 텍스트 데이터를 불러온다.
- Tokenizer로 모델 입력 형태로 바꾼다.
- 사전학습 한국어 언어모델을 분류 모델로 fine-tuning한다.
- 평가 지표, Confusion Matrix, Classification Report, 오답 샘플을 확인한다.
- 이 구조를 카페 리뷰 운영 신호 모델로 확장한다.

이번 최종본은 정확도 해석이 가능하도록 기본 학습 기준을 `3 epoch`로 설정했습니다.


In [1]:
# ============================================================
# 1-0. NSMC 안전 실행 초기화 셀
# ============================================================
# 이 셀의 목적:
# - 이후 NSMC 셀에서 필요한 기본 변수와 상태값을 한 번에 정의합니다.
# - 어느 셀부터 다시 실행해도 HF_READY, SEED, OUTPUT_DIR, LEARNING_RATE 등이 없어서 죽지 않게 합니다.
# - 기본값은 실제 평가용 학습 설정입니다. 빠른 구조 확인만 필요할 때만 SMOKE_TEST=True로 바꾸세요.

import os
import json
import time
import random
import inspect
import platform
from pathlib import Path

PROJECT_ROOT = Path.cwd()
NSMC_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "nsmc_intro_finetuning"
NSMC_REPORT_DIR = PROJECT_ROOT / "reports" / "nsmc_intro_finetuning"
NSMC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NSMC_REPORT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 핵심 학습 설정
# ------------------------------------------------------------
SEED = 42
MODEL_NAME = "klue/bert-base"
NUM_LABELS = 2
MAX_LENGTH = 128

# NSMC 실제 평가용 기본값
# 1 epoch는 smoke test용에 가깝기 때문에, 최종본에서는 3 epoch를 기본값으로 둡니다.
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32

# False: 전체 NSMC 데이터로 실제 학습
# True : 빠른 실행 확인용 일부 샘플만 사용
SMOKE_TEST = False
SMOKE_TRAIN_SIZE = 1200
SMOKE_EVAL_SIZE = 300

# 학습 실행 여부
RUN_NSMC_TRAINING = True
RUN_STEP5_BUCKETING = True

# 모든 NSMC 상태는 이 dict로 관리합니다.
NSMC_STATE = {
    "hf_ready": False,
    "dataset_ready": False,
    "model_ready": False,
    "tokenized_ready": False,
    "step4_done": False,
    "step5_done": False,
    "error_messages": [],
}

# 이후 셀에서 NameError가 나지 않도록 기본 객체를 먼저 만들어 둡니다.
dataset = None
train_df = None
test_df = None
tokenizer = None
model = None
model_bucket = None
tokenized_dataset = None
train_dataset = None
eval_dataset = None
valid_dataset = None  # 일부 기존 셀이 valid_dataset 이름을 참조해도 죽지 않도록 alias 준비
data_collator = None
trainer_step4 = None
trainer_step5 = None
step4_metrics = None
step5_metrics = None
comparison = None
sample_predictions = None
cm_df = None
report_df = None
wrong_df_sorted = None
label_map = {0: "negative", 1: "positive"}

try:
    import numpy as np
    import pandas as pd
    import torch
    from datasets import load_dataset, Dataset, DatasetDict
    from sklearn.metrics import (
        accuracy_score,
        precision_recall_fscore_support,
        classification_report,
        confusion_matrix,
    )
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        DataCollatorWithPadding,
        Trainer,
        TrainingArguments,
    )
    try:
        from transformers import set_seed as hf_set_seed
    except Exception:
        hf_set_seed = None

    HF_READY = True
    NSMC_STATE["hf_ready"] = True

except Exception as exc:
    HF_READY = False
    NSMC_STATE["hf_ready"] = False
    NSMC_STATE["error_messages"].append(f"라이브러리 import 실패: {repr(exc)}")
    print("NSMC 파트 필수 라이브러리 준비 실패")
    print("에러 내용:", repr(exc))
    print("필요 시 별도 셀에서 아래 설치 명령을 실행하세요:")
    print('!pip install -q "numpy==1.26.4" "pandas==2.2.2" "scikit-learn==1.5.2" "transformers==4.44.2" "datasets==2.21.0" "accelerate==0.34.2"')


def safe_display(obj):
    try:
        display(obj)
    except Exception:
        print(obj)


def set_global_seed(seed=42):
    random.seed(seed)
    if "np" in globals() and np is not None:
        np.random.seed(seed)
    if "torch" in globals() and torch is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    if "hf_set_seed" in globals() and hf_set_seed is not None:
        try:
            hf_set_seed(seed)
        except Exception:
            pass


def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def build_training_args(**kwargs):
    """Transformers 버전별 evaluation_strategy / eval_strategy 차이를 흡수합니다."""
    signature = inspect.signature(TrainingArguments.__init__)
    if "evaluation_strategy" not in signature.parameters and "evaluation_strategy" in kwargs:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
    return TrainingArguments(**kwargs)


def build_trainer(**kwargs):
    """Transformers 버전별 tokenizer / processing_class 차이를 흡수합니다."""
    signature = inspect.signature(Trainer.__init__)
    if "tokenizer" not in signature.parameters and "tokenizer" in kwargs:
        tokenizer_value = kwargs.pop("tokenizer")
        if "processing_class" in signature.parameters:
            kwargs["processing_class"] = tokenizer_value
    return Trainer(**kwargs)


def compute_warmup_steps(num_examples, batch_size, epochs, ratio=0.1):
    if num_examples is None or batch_size is None or epochs is None or batch_size <= 0:
        return 0
    steps_per_epoch = int(np.ceil(num_examples / batch_size))
    total_steps = steps_per_epoch * int(epochs)
    return max(1, int(total_steps * ratio))


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary",
        zero_division=0,
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }


def nsmc_ready_for_training():
    return bool(
        NSMC_STATE.get("hf_ready")
        and NSMC_STATE.get("dataset_ready")
        and NSMC_STATE.get("model_ready")
        and NSMC_STATE.get("tokenized_ready")
        and train_dataset is not None
        and eval_dataset is not None
        and tokenizer is not None
    )

set_global_seed(SEED)

print("Project root:", PROJECT_ROOT)
print("Python:", platform.python_version())
print("HF_READY:", HF_READY)
print("MODEL_NAME:", MODEL_NAME)
print("NUM_EPOCHS:", NUM_EPOCHS)
print("SMOKE_TEST:", SMOKE_TEST)
print("TRAIN_BATCH_SIZE:", TRAIN_BATCH_SIZE)
print("EVAL_BATCH_SIZE:", EVAL_BATCH_SIZE)
if HF_READY:
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
print("NSMC output dir:", NSMC_OUTPUT_DIR)


Project root: /workspace
Python: 3.11.10
HF_READY: True
MODEL_NAME: klue/bert-base
NUM_EPOCHS: 3
SMOKE_TEST: False
TRAIN_BATCH_SIZE: 16
EVAL_BATCH_SIZE: 32
PyTorch: 2.4.1+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 4090
NSMC output dir: /workspace/outputs/nsmc_intro_finetuning


## 1-1. NSMC 데이터셋 로드

NSMC는 HuggingFace 파인튜닝 구조를 검증하기 위한 선행 데이터입니다.

인터넷이나 HuggingFace Hub 접속이 되면 실제 NSMC 데이터를 사용하고, 실패하면 노트북 구조 확인용 미니 샘플 데이터로 fallback합니다.
이렇게 구성한 이유는 데이터 다운로드 실패 때문에 뒤쪽 카페 리뷰 서비스 파트까지 멈추는 것을 막기 위해서입니다.


In [2]:
# ============================================================
# 1-1. NSMC 데이터셋 로드
# ============================================================

if not HF_READY:
    print("NSMC 데이터 로드를 건너뜁니다: 필수 라이브러리가 준비되지 않았습니다.")
else:
    NSMC_DATA_FILES = {
        "train": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt",
        "test": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt",
    }

    try:
        dataset = load_dataset("nsmc")
        print("load_dataset('nsmc') 성공")

    except Exception as exc1:
        print("load_dataset('nsmc') 실패. 원본 TSV 파일 로드를 시도합니다.")
        print("1차 실패 원인:", repr(exc1))
        try:
            dataset = load_dataset("csv", data_files=NSMC_DATA_FILES, delimiter="\t")
            print("NSMC 원본 TSV fallback 성공")
        except Exception as exc2:
            print("외부 데이터 다운로드 실패. 구조 확인용 미니 샘플 데이터로 fallback합니다.")
            print("2차 실패 원인:", repr(exc2))
            mini_train = {
                "id": list(range(12)),
                "document": [
                    "정말 재미있고 감동적인 영화였습니다.",
                    "시간이 아까울 정도로 별로였어요.",
                    "배우 연기가 좋고 몰입감이 있었습니다.",
                    "스토리가 지루하고 결말도 아쉬웠습니다.",
                    "다시 보고 싶은 좋은 작품입니다.",
                    "추천하기 어려운 영화였습니다.",
                    "음악과 영상미가 훌륭했습니다.",
                    "전개가 산만해서 집중하기 힘들었습니다.",
                    "기대 이상으로 재미있었습니다.",
                    "내용이 너무 뻔하고 실망스러웠습니다.",
                    "커피 맛은 좋은데 직원 응대가 아쉬웠어요.",
                    "분위기가 좋아서 다시 방문하고 싶어요.",
                ],
                "label": [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1],
            }
            mini_test = {
                "id": list(range(100, 106)),
                "document": [
                    "정말 좋았습니다.",
                    "별로였습니다.",
                    "재미있게 봤어요.",
                    "지루했어요.",
                    "매장 분위기가 좋았습니다.",
                    "응대가 아쉬웠습니다.",
                ],
                "label": [1, 0, 1, 0, 1, 0],
            }
            dataset = DatasetDict({
                "train": Dataset.from_dict(mini_train),
                "test": Dataset.from_dict(mini_test),
            })

    # 결측 문장 제거
    dataset = dataset.filter(lambda example: example.get("document") is not None)

    train_df = dataset["train"].to_pandas()
    test_df = dataset["test"].to_pandas()
    NSMC_STATE["dataset_ready"] = True

    print("Train shape:", train_df.shape)
    print("Test shape:", test_df.shape)
    safe_display(train_df.head())


load_dataset('nsmc') 성공
Train shape: (150000, 3)
Test shape: (50000, 3)


,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


## 1-2. 데이터 기본 확인

여기서는 라벨 분포와 문장 길이를 확인합니다.
서비스 관점에서는 이 단계가 **데이터가 모델 학습에 들어갈 수 있는 상태인지 검수하는 구간**입니다.


In [3]:
# ============================================================
# 1-2. NSMC 데이터 기본 확인
# ============================================================

if not NSMC_STATE.get("dataset_ready"):
    print("데이터 기본 확인을 건너뜁니다: dataset이 준비되지 않았습니다.")
else:
    summary = pd.DataFrame({
        "split": ["train", "test"],
        "rows": [len(train_df), len(test_df)],
        "null_document": [train_df["document"].isna().sum(), test_df["document"].isna().sum()],
        "duplicate_document": [train_df["document"].duplicated().sum(), test_df["document"].duplicated().sum()],
    })
    safe_display(summary)

    print("Train label distribution")
    safe_display(train_df["label"].value_counts().sort_index().rename_axis("label").reset_index(name="count"))

    print("Test label distribution")
    safe_display(test_df["label"].value_counts().sort_index().rename_axis("label").reset_index(name="count"))

    for split_name, df in [("train", train_df), ("test", test_df)]:
        df["text_len"] = df["document"].fillna("").astype(str).apply(len)
        print(f"{split_name} text length statistics")
        safe_display(df["text_len"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))


,split,rows,null_document,duplicate_document
0,train,150000,0,3817
1,test,50000,0,842


Train label distribution


,label,count
0,0,75173
1,1,74827


Test label distribution


,label,count
0,0,24827
1,1,25173


train text length statistics


count    150000.000000
mean         35.203353
std          29.532097
min           0.000000
50%          27.000000
75%          42.000000
90%          75.000000
95%         107.000000
99%         139.000000
max         146.000000
Name: text_len, dtype: float64

test text length statistics


count    50000.000000
mean        35.318140
std         29.648683
min          0.000000
50%         27.000000
75%         43.000000
90%         76.000000
95%        108.000000
99%        139.000000
max        144.000000
Name: text_len, dtype: float64

## 1-3. Tokenizer와 모델 로드

`klue/bert-base`를 사용해 한국어 문장을 모델 입력으로 바꿉니다.
모델 다운로드가 실패하면 이후 학습 셀은 자동으로 스킵됩니다.


In [4]:
# ============================================================
# 1-3. Tokenizer와 Model 로드
# ============================================================

if not HF_READY:
    print("Tokenizer/Model 로드를 건너뜁니다: 필수 라이브러리가 준비되지 않았습니다.")
else:
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
        data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
        NSMC_STATE["model_ready"] = True

        print("Tokenizer vocab size:", getattr(tokenizer, "vocab_size", "unknown"))
        print("Model type:", model.config.model_type)
        print("Hidden size:", model.config.hidden_size)
        print("Classification labels:", model.config.num_labels)

    except Exception as exc:
        NSMC_STATE["model_ready"] = False
        NSMC_STATE["error_messages"].append(f"모델/토크나이저 로드 실패: {repr(exc)}")
        print("Tokenizer/Model 로드 실패")
        print("에러 내용:", repr(exc))
        print("인터넷 연결 또는 HuggingFace Hub 접근 권한을 확인하세요.")


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenizer vocab size: 32000
Model type: bert
Hidden size: 768
Classification labels: 2


## 1-4. Tokenization과 학습 데이터 구성

문장을 토큰 ID로 변환하고, 학습/평가 데이터셋을 구성합니다.

기본값은 `SMOKE_TEST=False`이므로 전체 NSMC 데이터를 사용합니다.  
실행 시간만 확인해야 하는 경우에만 초기화 셀에서 `SMOKE_TEST=True`로 바꾸면 일부 샘플만 사용합니다.


In [5]:
# ============================================================
# 1-4. Tokenization과 train/eval 데이터 구성
# ============================================================

if not (NSMC_STATE.get("dataset_ready") and NSMC_STATE.get("model_ready")):
    print("Tokenization을 건너뜁니다: dataset 또는 model/tokenizer가 준비되지 않았습니다.")
else:
    try:
        def tokenize_function(batch):
            return tokenizer(
                batch["document"],
                truncation=True,
                max_length=MAX_LENGTH,
            )

        tokenized_dataset = dataset.map(tokenize_function, batched=True)
        remove_cols = [col for col in ["id", "document"] if col in tokenized_dataset["train"].column_names]
        if remove_cols:
            tokenized_dataset = tokenized_dataset.remove_columns(remove_cols)
        tokenized_dataset.set_format("torch")

        if SMOKE_TEST:
            train_size = min(SMOKE_TRAIN_SIZE, len(tokenized_dataset["train"]))
            eval_size = min(SMOKE_EVAL_SIZE, len(tokenized_dataset["test"]))
            train_dataset = tokenized_dataset["train"].shuffle(seed=SEED).select(range(train_size))
            eval_dataset = tokenized_dataset["test"].shuffle(seed=SEED).select(range(eval_size))
        else:
            train_dataset = tokenized_dataset["train"]
            eval_dataset = tokenized_dataset["test"]

        valid_dataset = eval_dataset  # 기존 코드 호환용 alias
        NSMC_STATE["tokenized_ready"] = True

        print("Tokenized dataset ready")
        print("Train rows:", len(train_dataset))
        print("Eval rows:", len(eval_dataset))
        print("Training mode:", "SMOKE_TEST 일부 샘플" if SMOKE_TEST else "FULL NSMC 전체 데이터")
        print("Epochs:", NUM_EPOCHS)
        print("Train batch size:", TRAIN_BATCH_SIZE)
        print("Eval batch size:", EVAL_BATCH_SIZE)

    except Exception as exc:
        NSMC_STATE["tokenized_ready"] = False
        NSMC_STATE["error_messages"].append(f"Tokenization 실패: {repr(exc)}")
        print("Tokenization 실패")
        print("에러 내용:", repr(exc))


Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Tokenized dataset ready
Train rows: 150000
Eval rows: 50000
Training mode: FULL NSMC 전체 데이터
Epochs: 3
Train batch size: 16
Eval batch size: 32


## 1-5. STEP 4: 기본 Fine-tuning

STEP 4는 기본 Trainer 설정으로 `klue/bert-base`를 NSMC 감성분류에 맞게 fine-tuning합니다.

이 최종본에서는 정확도 해석이 가능하도록 기본값을 `NUM_EPOCHS=3`으로 설정했습니다.  
따라서 실행 시간은 길어질 수 있지만, 단순 smoke test보다 훨씬 설득력 있는 평가 결과를 얻는 것이 목적입니다.


In [6]:
# ============================================================
# 1-5. STEP 4 기본 Fine-tuning
# ============================================================

if not RUN_NSMC_TRAINING:
    print("RUN_NSMC_TRAINING=False 이므로 STEP 4 학습을 건너뜁니다.")

elif not nsmc_ready_for_training():
    print("STEP 4 학습을 실행하지 않았습니다.")
    print("현재 NSMC_STATE:")
    safe_display(NSMC_STATE)
    print("필요 조건: hf_ready, dataset_ready, model_ready, tokenized_ready가 모두 True여야 합니다.")

else:
    try:
        set_global_seed(SEED)
        WARMUP_STEPS = compute_warmup_steps(len(train_dataset), TRAIN_BATCH_SIZE, NUM_EPOCHS, ratio=0.1)
        print("Warmup steps:", WARMUP_STEPS)

        training_args_step4 = build_training_args(
            output_dir=str(NSMC_OUTPUT_DIR / "step4_klue_bert_nsmc"),
            learning_rate=LEARNING_RATE,
            per_device_train_batch_size=TRAIN_BATCH_SIZE,
            per_device_eval_batch_size=EVAL_BATCH_SIZE,
            num_train_epochs=NUM_EPOCHS,
            weight_decay=0.01,
            warmup_steps=WARMUP_STEPS,
            evaluation_strategy="epoch",
            save_strategy="epoch",
            logging_strategy="steps",
            logging_steps=100,
            load_best_model_at_end=True,
            metric_for_best_model="accuracy",
            greater_is_better=True,
            fp16=torch.cuda.is_available(),
            report_to="none",
            seed=SEED,
            data_seed=SEED,
        )

        trainer_step4 = build_trainer(
            model=model,
            args=training_args_step4,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            tokenizer=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
        )

        start_time = time.time()
        trainer_step4.train()
        step4_time = time.time() - start_time
        step4_metrics = trainer_step4.evaluate()
        step4_metrics["training_time_sec"] = float(step4_time)
        step4_metrics["max_length"] = MAX_LENGTH
        step4_metrics["train_batch_size"] = TRAIN_BATCH_SIZE
        step4_metrics["eval_batch_size"] = EVAL_BATCH_SIZE
        step4_metrics["num_train_epochs"] = NUM_EPOCHS
        step4_metrics["group_by_length"] = False
        NSMC_STATE["step4_done"] = True

        safe_display(step4_metrics)

    except Exception as exc:
        NSMC_STATE["step4_done"] = False
        NSMC_STATE["error_messages"].append(f"STEP4 학습 실패: {repr(exc)}")
        print("STEP 4 학습 중 문제가 발생했습니다. 노트북 전체 실행은 중단하지 않습니다.")
        print("에러 내용:", repr(exc))


Warmup steps: 2812


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.262300,0.245634,0.902820,0.894293,0.915147,0.904600
2,0.207000,0.256776,0.906680,0.909665,0.904461,0.907055
3,0.143100,0.360570,0.906940,0.901408,0.915266,0.908285


{'eval_loss': 0.36057019233703613,
 'eval_accuracy': 0.90694,
 'eval_precision': 0.9014084507042254,
 'eval_recall': 0.9152663568108688,
 'eval_f1': 0.9082845485187157,
 'eval_runtime': 22.1254,
 'eval_samples_per_second': 2259.845,
 'eval_steps_per_second': 70.643,
 'epoch': 3.0,
 'training_time_sec': 2327.40771651268,
 'max_length': 128,
 'train_batch_size': 16,
 'eval_batch_size': 32,
 'num_train_epochs': 3,
 'group_by_length': False}

## 1-6. STEP 4 결과 저장

학습이 정상 완료된 경우 평가 결과를 JSON으로 저장합니다.
학습이 스킵되었거나 실패한 경우에는 안내만 출력하고 넘어갑니다.


In [7]:
# ============================================================
# 1-6. STEP 4 결과 저장
# ============================================================

if step4_metrics is None:
    print("STEP 4 평가 결과가 없어 저장을 건너뜁니다.")
    print("앞 셀에서 STEP 4가 정상 완료되면 metrics_step4.json이 저장됩니다.")
else:
    save_json(step4_metrics, NSMC_OUTPUT_DIR / "metrics_step4.json")
    print("Saved:", NSMC_OUTPUT_DIR / "metrics_step4.json")


Saved: /workspace/outputs/nsmc_intro_finetuning/metrics_step4.json


## 1-7. STEP 5: Bucketing + Dynamic Padding 적용

STEP 5는 `group_by_length=True`를 적용해 비슷한 길이의 문장끼리 묶어 학습 효율을 높이는 실험입니다.
서비스 관점에서는 같은 모델이라도 **학습 효율과 운영 비용을 줄이는 실험**으로 볼 수 있습니다.


In [8]:
# ============================================================
# 1-7. STEP 5 Bucketing 적용 Fine-tuning
# ============================================================

if not RUN_NSMC_TRAINING or not RUN_STEP5_BUCKETING:
    print("RUN_NSMC_TRAINING=False 또는 RUN_STEP5_BUCKETING=False 이므로 STEP 5 학습을 건너뜁니다.")

elif not nsmc_ready_for_training():
    print("STEP 5 학습을 실행하지 않았습니다.")
    print("현재 NSMC_STATE:")
    safe_display(NSMC_STATE)

else:
    try:
        set_global_seed(SEED)
        WARMUP_STEPS = compute_warmup_steps(len(train_dataset), TRAIN_BATCH_SIZE, NUM_EPOCHS, ratio=0.1)
        model_bucket = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

        training_args_step5 = build_training_args(
            output_dir=str(NSMC_OUTPUT_DIR / "step5_klue_bert_nsmc_bucket"),
            learning_rate=LEARNING_RATE,
            per_device_train_batch_size=TRAIN_BATCH_SIZE,
            per_device_eval_batch_size=EVAL_BATCH_SIZE,
            num_train_epochs=NUM_EPOCHS,
            weight_decay=0.01,
            warmup_steps=WARMUP_STEPS,
            evaluation_strategy="epoch",
            save_strategy="epoch",
            logging_strategy="steps",
            logging_steps=100,
            load_best_model_at_end=True,
            metric_for_best_model="accuracy",
            greater_is_better=True,
            fp16=torch.cuda.is_available(),
            group_by_length=True,
            report_to="none",
            seed=SEED,
            data_seed=SEED,
        )

        trainer_step5 = build_trainer(
            model=model_bucket,
            args=training_args_step5,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            tokenizer=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
        )

        start_time = time.time()
        trainer_step5.train()
        step5_time = time.time() - start_time
        step5_metrics = trainer_step5.evaluate()
        step5_metrics["training_time_sec"] = float(step5_time)
        step5_metrics["max_length"] = MAX_LENGTH
        step5_metrics["train_batch_size"] = TRAIN_BATCH_SIZE
        step5_metrics["eval_batch_size"] = EVAL_BATCH_SIZE
        step5_metrics["num_train_epochs"] = NUM_EPOCHS
        step5_metrics["group_by_length"] = True
        NSMC_STATE["step5_done"] = True

        safe_display(step5_metrics)

    except Exception as exc:
        NSMC_STATE["step5_done"] = False
        NSMC_STATE["error_messages"].append(f"STEP5 학습 실패: {repr(exc)}")
        print("STEP 5 학습 중 문제가 발생했습니다. 노트북 전체 실행은 중단하지 않습니다.")
        print("에러 내용:", repr(exc))


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.272700,0.245612,0.898940,0.890104,0.911850,0.900846
2,0.199300,0.278582,0.904940,0.919854,0.888611,0.903962
3,0.142100,0.350541,0.906560,0.903420,0.911890,0.907635


{'eval_loss': 0.35054144263267517,
 'eval_accuracy': 0.90656,
 'eval_precision': 0.9034200480144831,
 'eval_recall': 0.911889723116037,
 'eval_f1': 0.9076351271203195,
 'eval_runtime': 22.1579,
 'eval_samples_per_second': 2256.53,
 'eval_steps_per_second': 70.539,
 'epoch': 3.0,
 'training_time_sec': 2318.9611370563507,
 'max_length': 128,
 'train_batch_size': 16,
 'eval_batch_size': 32,
 'num_train_epochs': 3,
 'group_by_length': True}

## 1-8. STEP 5 결과 저장과 STEP 4/5 비교

두 실험 결과가 모두 있으면 비교표를 생성합니다.
둘 중 하나라도 없으면 가능한 결과만 저장하고, 비교는 건너뜁니다.


In [9]:
# ============================================================
# 1-8. STEP 5 저장 및 STEP 4/5 비교
# ============================================================

def normalize_metrics(name, metrics):
    if metrics is None:
        return None
    return {
        "experiment": name,
        "accuracy": metrics.get("eval_accuracy"),
        "eval_loss": metrics.get("eval_loss"),
        "precision": metrics.get("eval_precision"),
        "recall": metrics.get("eval_recall"),
        "f1": metrics.get("eval_f1"),
        "training_time_sec": metrics.get("training_time_sec"),
        "group_by_length": metrics.get("group_by_length"),
    }

if step5_metrics is not None:
    save_json(step5_metrics, NSMC_OUTPUT_DIR / "metrics_step5_bucketing.json")
    print("Saved:", NSMC_OUTPUT_DIR / "metrics_step5_bucketing.json")
else:
    print("STEP 5 평가 결과가 없어 저장을 건너뜁니다.")

rows = []
for name, metrics in [
    ("STEP4_finetuning", step4_metrics),
    ("STEP5_bucketing_dynamic_padding", step5_metrics),
]:
    row = normalize_metrics(name, metrics)
    if row is not None:
        rows.append(row)

if len(rows) > 0:
    comparison = pd.DataFrame(rows)
    if "training_time_sec" in comparison.columns:
        comparison["training_time_min"] = comparison["training_time_sec"] / 60
    safe_display(comparison)
    comparison.to_csv(NSMC_OUTPUT_DIR / "step4_vs_step5_comparison.csv", index=False, encoding="utf-8-sig")
    print("Saved:", NSMC_OUTPUT_DIR / "step4_vs_step5_comparison.csv")
else:
    print("비교할 STEP 4/5 결과가 아직 없습니다.")


Saved: /workspace/outputs/nsmc_intro_finetuning/metrics_step5_bucketing.json


,experiment,accuracy,eval_loss,precision,recall,f1,training_time_sec,group_by_length,training_time_min
0,STEP4_finetuning,0.90694,0.360570,0.901408,0.915266,0.908285,2327.407717,False,38.790129
1,STEP5_bucketing_dynamic_padding,0.90656,0.350541,0.903420,0.911890,0.907635,2318.961137,True,38.649352


Saved: /workspace/outputs/nsmc_intro_finetuning/step4_vs_step5_comparison.csv


## 1-9. 샘플 문장 예측

학습된 STEP 5 모델이 있으면 샘플 문장을 예측합니다.
STEP 5가 없고 STEP 4만 있으면 STEP 4 모델을 사용합니다.


In [10]:
# ============================================================
# 1-9. 샘플 문장 예측
# ============================================================

active_trainer = trainer_step5 if trainer_step5 is not None else trainer_step4
label_map = {0: "negative", 1: "positive"}

if active_trainer is None or tokenizer is None:
    print("예측 가능한 trainer/tokenizer가 없어 샘플 예측을 건너뜁니다.")
else:
    try:
        texts = [
            "정말 재미있고 감동적인 영화였습니다.",
            "시간이 아까울 정도로 별로였어요.",
            "커피 맛은 좋은데 직원 응대가 조금 아쉬웠어요.",
            "분위기가 좋아서 다시 방문하고 싶어요.",
        ]

        inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH)
        inputs = {key: value.to(active_trainer.model.device) for key, value in inputs.items()}

        active_trainer.model.eval()
        with torch.no_grad():
            logits = active_trainer.model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            preds = np.argmax(probs, axis=-1)

        sample_predictions = pd.DataFrame({
            "text": texts,
            "prediction": preds,
            "prediction_label": [label_map[int(pred)] for pred in preds],
            "negative_prob": probs[:, 0],
            "positive_prob": probs[:, 1],
        })

        safe_display(sample_predictions)
        sample_predictions.to_csv(NSMC_OUTPUT_DIR / "sample_predictions.csv", index=False, encoding="utf-8-sig")
        print("Saved:", NSMC_OUTPUT_DIR / "sample_predictions.csv")

    except Exception as exc:
        print("샘플 예측 중 문제가 발생했습니다. 노트북 전체 실행은 중단하지 않습니다.")
        print("에러 내용:", repr(exc))


,text,prediction,prediction_label,negative_prob,positive_prob
0,정말 재미있고 감동적인 영화였습니다.,1,positive,0.012528,0.987472
1,시간이 아까울 정도로 별로였어요.,0,negative,0.999573,0.000427
2,커피 맛은 좋은데 직원 응대가 조금 아쉬웠어요.,1,positive,0.237134,0.762866
3,분위기가 좋아서 다시 방문하고 싶어요.,1,positive,0.004199,0.995801


Saved: /workspace/outputs/nsmc_intro_finetuning/sample_predictions.csv


## 1-10. Confusion Matrix, Classification Report, 오답 분석

최종 평가 데이터 기준으로 예측 결과를 만들고, 혼동행렬·분류 리포트·오답 샘플을 저장합니다.
이 결과는 이후 카페 리뷰 운영 신호 모델에서도 같은 방식으로 평가 구조를 재사용할 수 있습니다.


In [11]:
# ============================================================
# 1-10. Confusion Matrix / Classification Report / 오답 분석
# ============================================================

active_trainer = trainer_step5 if trainer_step5 is not None else trainer_step4

if active_trainer is None or eval_dataset is None:
    print("평가 가능한 trainer/eval_dataset이 없어 상세 평가를 건너뜁니다.")
else:
    try:
        predictions_output = active_trainer.predict(eval_dataset)
        logits_all = predictions_output.predictions
        preds_all = np.argmax(logits_all, axis=-1)
        labels_all = predictions_output.label_ids

        cm = confusion_matrix(labels_all, preds_all)
        cm_df = pd.DataFrame(
            cm,
            index=["실제_부정", "실제_긍정"],
            columns=["예측_부정", "예측_긍정"],
        )
        safe_display(cm_df)
        cm_df.to_csv(NSMC_OUTPUT_DIR / "confusion_matrix.csv", encoding="utf-8-sig")

        report = classification_report(
            labels_all,
            preds_all,
            target_names=["negative", "positive"],
            output_dict=True,
            zero_division=0,
        )
        report_df = pd.DataFrame(report).transpose()
        safe_display(report_df)
        report_df.to_csv(NSMC_OUTPUT_DIR / "classification_report.csv", encoding="utf-8-sig")
        save_json(report, NSMC_OUTPUT_DIR / "classification_report.json")

        if test_df is not None:
            probs_all = torch.softmax(torch.tensor(logits_all), dim=-1).numpy()
            wrong_mask = preds_all != labels_all
            wrong_indices = np.where(wrong_mask)[0]
            base_eval_df = test_df.iloc[:len(labels_all)].reset_index(drop=True)

            wrong_df_sorted = base_eval_df.iloc[wrong_indices][["id", "document", "label"]].copy()
            wrong_df_sorted["pred"] = preds_all[wrong_indices]
            wrong_df_sorted["true_label"] = wrong_df_sorted["label"].map(label_map)
            wrong_df_sorted["pred_label"] = wrong_df_sorted["pred"].map(label_map)
            wrong_df_sorted["negative_prob"] = probs_all[wrong_indices, 0]
            wrong_df_sorted["positive_prob"] = probs_all[wrong_indices, 1]
            wrong_df_sorted["confidence"] = probs_all[wrong_indices].max(axis=1)
            wrong_df_sorted = wrong_df_sorted.sort_values("confidence", ascending=False)

            safe_display(wrong_df_sorted.head(30))
            wrong_df_sorted.to_csv(NSMC_OUTPUT_DIR / "wrong_predictions.csv", index=False, encoding="utf-8-sig")
            print("오답 개수:", len(wrong_df_sorted))

        print("Saved detailed NSMC evaluation files to:", NSMC_OUTPUT_DIR)

    except Exception as exc:
        print("상세 평가 중 문제가 발생했습니다. 노트북 전체 실행은 중단하지 않습니다.")
        print("에러 내용:", repr(exc))


,예측_부정,예측_긍정
실제_부정,22373,2454
실제_긍정,2218,22955


,precision,recall,f1-score,support
negative,0.909804,0.901156,0.905460,24827.00000
positive,0.903420,0.911890,0.907635,25173.00000
accuracy,0.906560,0.906560,0.906560,0.90656
macro avg,0.906612,0.906523,0.906547,50000.00000
weighted avg,0.906590,0.906560,0.906555,50000.00000


,id,document,label,pred,true_label,pred_label,negative_prob,positive_prob,confidence
9140,9014317,많은 생각을 하게 만드네요... 모든 순간이 선택이겠지만.. 줄리엣 비노쉬의 명연기...,0,1,negative,positive,0.000233,0.999767,0.999767
26803,9652381,설정도 어의없고 김꽃비씨 연제욱씨 두분연기도 못봐주겠고 김꽃비씨 대사 치는거 대박 ...,1,0,positive,negative,0.999706,0.000294,0.999706
6081,6672788,다른 시시껍적한 블록버스터보다 더 좋고 생각많이 하게 하는 영화. 형이 푸켓에 7년...,0,1,negative,positive,0.000297,0.999703,0.999703
1226,9848365,아낙 바하라히 헤이난코텍 테라 카포 이거 한 200번들렸다ㅋ 2천원 주고다운받았는데...,0,1,negative,positive,0.000298,0.999702,0.999702
11491,8883562,다 떠나서 중금속+방사능 콤보인 맛이라곤 눈곱만큼도 없는걸 먹겠다고 저딴 짓거리를 ...,1,0,positive,negative,0.999681,0.000319,0.999681
35831,6975087,완전 실망했다는 평가가 참 많네.. 무슨 쫄깃한 로맨스라도 기대했는건가? 그리고 '...,1,0,positive,negative,0.999662,0.000338,0.999662
41750,7233427,신박한 영화. 모티브가 코미디지만 웃기거나 하진 않았고... 하지만 묘하게 몰입되서...,0,1,negative,positive,0.000355,0.999645,0.999645
8777,9591319,후. . .노답임. .정말노답 ★,1,0,positive,negative,0.999628,0.000372,0.999628
49701,4384978,제목에 설레였었던 추억이.. 나와야 할 씬이 안 나와서 실망했었지만 즐거웠던 영화..,0,1,negative,positive,0.000380,0.999620,0.999620
20605,8719753,재미없다.잘키운 딸하나 많은 시청보지마세요.,1,0,positive,negative,0.999601,0.000399,0.999601


오답 개수: 4672
Saved detailed NSMC evaluation files to: /workspace/outputs/nsmc_intro_finetuning


## 1-11. NSMC 파트 정리

NSMC 파트의 의미는 다음과 같습니다.

- 감성분석 자체가 최종 목적은 아닙니다.
- HuggingFace 모델을 커스텀 데이터에 연결하는 기본 구조를 실제 fine-tuning 기준으로 검증하는 것이 목적입니다.
- `1 epoch`가 아니라 `3 epoch`를 기본값으로 사용해 평가 지표를 해석할 수 있게 했습니다.
- 이후 같은 구조를 카페 리뷰 데이터에 적용하면, 긍정/부정이 아니라 `맛`, `응대`, `가격`, `재방문`, `대기`, `매장 분위기` 같은 운영 신호를 추출할 수 있습니다.
- 이 운영 신호가 누적되면 매장별 Store DNA를 만들 수 있습니다.
- Store DNA는 최종적으로 사장님에게 “현재 상태”와 “내일 할 일”을 알려주는 서비스 엔진으로 연결됩니다.


---

# Part 2. 카페 리뷰 운영 신호 → Store DNA → 서비스화

이제부터는 기존 파이널 노트북의 본문입니다.
앞의 NSMC 파트에서 확인한 HuggingFace 파인튜닝 구조를 바탕으로, 카페 리뷰 운영 신호 모델과 서비스화 구조로 확장합니다.



# HuggingFace 커스텀 서비스 프레임워크 최종 노트북  
## NSMC → 카페 리뷰 운영 신호 → Store DNA → 서비스화

이 노트북의 목적은 단순히 “카페 리뷰 모델 하나”를 만드는 것이 아닙니다.

> **HuggingFace Transformers Framework 안의 Model, Tokenizer, Trainer 구조를 이해하고,  
> 어떤 도메인의 서비스 문제에도 같은 절차로 커스터마이징할 수 있는 나만의 실전 프레임워크를 만드는 것**입니다.

이번 프로젝트에서는 그 예시로 **카페 리뷰를 사장님용 운영 신호와 Store DNA 리포트로 바꾸는 서비스**를 만들었습니다.

전체 흐름은 다음과 같습니다.

```text
1. NSMC 감성분석 실습
   → HuggingFace Tokenizer / Model / Trainer 기본 구조 이해

2. 카페 리뷰 단일 라벨 분류
   → 리뷰를 긍정/부정이 아니라 운영 신호로 분류

3. 카페 리뷰 다중 라벨 분류 v1.2
   → 실제 리뷰처럼 한 문장에 여러 운영 의미가 동시에 담기는 구조 반영

4. 모델 저장
   → config.json, model.safetensors 또는 pytorch_model.bin, tokenizer 파일 생성

5. 실제 카페 리뷰 1,000개 적용
   → 학습된 모델을 서비스 추론 파이프라인으로 사용

6. Store DNA / 사장님 리포트 생성
   → 모델 출력값을 서비스 언어로 변환

7. Codex / FastAPI 서비스화
   → 노트북 코드를 API와 대시보드로 분리
```

---

## 이 노트북을 다른 서비스에 재사용하는 방법

카페 리뷰가 아니더라도 아래 8가지만 바꾸면 같은 구조를 다시 쓸 수 있습니다.

| 프레임워크 단계 | 이번 프로젝트 | 다른 서비스로 바꿀 때 |
|---|---|---|
| Problem | 카페 리뷰 운영 신호 분류 | 고객 문의 분류, 상담 품질 분석, 병원 후기 분석, 민원 분류 등 |
| Dataset | 카페 리뷰 문장 | 해당 서비스의 텍스트 데이터 |
| Label | 제품/서비스/가격/공간/재방문 신호 | 서비스 목적에 맞는 라벨 체계 |
| Tokenizer | `klue/bert-base` tokenizer | 한국어/영어/도메인에 맞는 tokenizer |
| Model | 다중 라벨 BERT 분류기 | 분류/회귀/생성/검색 모델 |
| Trainer | HuggingFace Trainer | 학습 루프 또는 PEFT/LoRA 등 |
| Metrics | micro/macro F1, precision, recall | 서비스 품질 기준에 맞는 지표 |
| Service Output | Store DNA, 사장님 액션 | 사용자가 이해하는 최종 리포트/추천/판정 |



# 0. 커스텀 프로젝트 설계 프레임워크

이 노트북의 핵심은 아래 구조를 반복해서 사용할 수 있게 만드는 것입니다.

```text
Raw Text
→ Label System
→ Dataset
→ Tokenizer
→ Model
→ Trainer
→ Evaluation
→ Save Artifacts
→ Inference Pipeline
→ Service Output
```

HuggingFace에서 특히 중요한 객체는 3개입니다.

```text
Tokenizer
- 사람의 문장을 모델이 이해할 수 있는 input_ids / attention_mask로 바꾼다.

Model
- 토큰화된 입력을 받아 라벨별 점수(logits)를 만든다.

Trainer
- 데이터셋, 모델, 평가 지표, 학습 설정을 묶어서 fine-tuning을 수행한다.
```

서비스화를 위해서는 여기에 3개가 더 필요합니다.

```text
Label Map
- 모델의 숫자 출력이 어떤 서비스 의미인지 연결한다.

Threshold
- 다중 라벨에서 어떤 확률 이상을 라벨로 볼지 결정한다.

Report Layer
- 모델 출력을 사용자가 이해하는 문장과 액션으로 바꾼다.
```



# 1. 프로젝트 발전 흐름

## 1-1. NSMC 감성분석에서 얻은 것

처음에는 NSMC 영화 리뷰 감성분석으로 출발했습니다.  
이 단계의 목적은 모델 성능 자체보다 HuggingFace의 기본 작업 순서를 익히는 것이었습니다.

```text
문장
→ Tokenizer
→ Dataset
→ Model
→ Trainer
→ Evaluation
```

NSMC에서는 긍정/부정이라는 단순한 라벨을 사용했습니다.

## 1-2. 카페 리뷰 단일 라벨 분류

그 다음에는 카페 리뷰를 긍정/부정이 아니라 운영 신호로 바꾸었습니다.

```text
"커피는 맛있는데 가격이 조금 비싸요."
```

이 문장을 단순히 부정으로 보지 않고,

```text
제품/메뉴 강점 + 가격 저항
```

으로 해석하는 방향입니다.

## 1-3. 다중 라벨 분류 v1.2

실제 리뷰는 하나의 의미만 담지 않습니다.  
그래서 단일 라벨 분류에서 다중 라벨 분류로 확장했습니다.

예시:

```text
"디저트는 맛있고 분위기도 좋은데 웨이팅이 길었어요."
→ 제품/메뉴 강점
→ 공간/분위기 강점
→ 대기/혼잡 리스크
```

이 구조가 실제 서비스에 더 적합합니다.



# 2. 실행 방법

이 노트북은 두 가지 모드로 사용할 수 있습니다.

## A. 모델 학습까지 다시 실행하는 모드

아래 학습 셀들을 순서대로 실행하면 모델이 저장됩니다.

생성되는 핵심 파일:

```text
models/cafe_review_signal_multilabel_v1_2/
├─ config.json
├─ model.safetensors 또는 pytorch_model.bin
├─ tokenizer.json
├─ tokenizer_config.json
├─ special_tokens_map.json
├─ metadata.json
├─ operation_labels.json
├─ label_thresholds.json
├─ label2id.json
└─ id2label.json
```

## B. 이미 저장된 모델로 실제 리뷰만 분석하는 모드

학습을 다시 하지 않고, 아래 후반부의 **저장 모델 로드 → 실제 리뷰 1,000개 분석 → Store DNA 생성** 부분만 실행하면 됩니다.

서비스화 단계에서는 B 모드가 중요합니다.


# 3. 카페 리뷰 운영 신호 다중 라벨 모델 학습 파트

아래 섹션은 기존 `cafe_review_v1_2_FINAL_model_save_ready` 노트북의 핵심 학습 파이프라인입니다.

목표는 다중 라벨 모델을 학습하고, 서비스에서 로드할 수 있는 HuggingFace 모델 산출물을 저장하는 것입니다.

## 0. 실행 순서

1. 패키지 설치 셀 실행
2. **Kernel Restart / 런타임 다시 시작**
3. 노트북을 처음부터 다시 실행
4. 학습 완료 후 threshold 튜닝 결과 확인
5. best threshold / label-wise threshold / top-k 결과 비교
6. 샘플 리뷰 예측과 Store DNA 점수 확인

이 노트북은 checkpoint 저장 오류를 피하기 위해 `save_strategy="no"`를 사용합니다.

## 1. 안정 패키지 설치 셀

RunPod RTX 4090 / CUDA 12.4 기준 안정 조합입니다. 설치 후 반드시 커널을 재시작하세요.

In [12]:
# RunPod RTX 4090 / CUDA 12.4 기준 안정 설치 셀
# 설치 후 반드시 Kernel Restart / 런타임 다시 시작을 하세요.

!pip uninstall -y transformers torch torchvision torchaudio accelerate datasets evaluate -q
!pip install -q torch==2.4.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate==0.33.0 evaluate scikit-learn pandas numpy matplotlib

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


## 2. 라이브러리 불러오기와 실행 환경 확인

이 셀은 실제 GPU와 패키지 버전을 확인합니다.

In [13]:
import os, re, json, time, random, shutil, ast
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import transformers
import datasets
import accelerate

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs" / "cafe_review_signal_multilabel_v1_2"
MODEL_DIR = BASE_DIR / "models" / "cafe_review_signal_multilabel_v1_2"
SERVICE_DIR = BASE_DIR / "app"
for p in [DATA_DIR, OUTPUT_DIR, MODEL_DIR, SERVICE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch: 2.4.1+cu124
transformers: 4.44.2
datasets: 2.21.0
accelerate: 0.33.0
CUDA 사용 가능: True
GPU: NVIDIA GeForce RTX 4090


## 3. 디스크 용량 확인과 checkpoint 정리

이전 실험에서 생긴 checkpoint와 임시 파일이 디스크를 차지할 수 있으므로 정리합니다.

In [14]:
import os
from pathlib import Path

PROJECT_NAME = "cafe_review_signal_multilabel_v1_2"
PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / PROJECT_NAME
MODEL_DIR = PROJECT_ROOT / "models" / PROJECT_NAME

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("프로젝트 기본 경로 설정 완료")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"MODEL_DIR: {MODEL_DIR}")

print("\n현재 작업 폴더 파일 목록:")
for item in sorted(PROJECT_ROOT.iterdir()):
    print("-", item.name)

프로젝트 기본 경로 설정 완료
PROJECT_ROOT: /workspace
DATA_DIR: /workspace/data
OUTPUT_DIR: /workspace/outputs/cafe_review_signal_multilabel_v1_2
MODEL_DIR: /workspace/models/cafe_review_signal_multilabel_v1_2

현재 작업 폴더 파일 목록:
- .ipynb_checkpoints
- all_cafe_reviews_1000.csv
- app
- cafe_review_final_with_nsmc_front.ipynb
- cafe_review_final_with_nsmc_story.ipynb
- cafe_review_v1_2_FINAL_model_save_ready.ipynb
- cafe_review_v1_2_FINAL_model_save_ready_with_nsmc_intro.ipynb
- cafe_review_v1_2_FINAL_model_save_ready_with_nsmc_intro_fixed.ipynb
- data
- huggingface_custom_service_framework_FINAL.ipynb
- huggingface_custom_service_framework_FINAL_v2_error_safe.ipynb
- huggingface_custom_service_framework_FINAL_v3_real_training.ipynb
- models
- outputs
- reports


## 4. 운영 신호 라벨 정의

서비스에서 사용할 운영 신호 라벨입니다. 이번 노트북은 이 라벨들을 **다중 라벨**로 예측합니다.

In [15]:
OPERATION_LABELS = [
    "product_strength",
    "product_risk",
    "service_strength",
    "service_risk",
    "price_resistance",
    "price_value_strength",
    "waiting_risk",
    "space_strength",
    "space_risk",
    "cleanliness_strength",
    "cleanliness_risk",
    "revisit_intent",
    "revisit_risk",
    "operation_efficiency",
]

LABEL_KO = {
    "product_strength": "제품 경쟁력",
    "product_risk": "제품 품질 리스크",
    "service_strength": "서비스 강점",
    "service_risk": "서비스 리스크",
    "price_resistance": "가격 저항",
    "price_value_strength": "가격 대비 만족",
    "waiting_risk": "대기/혼잡 리스크",
    "space_strength": "공간 강점",
    "space_risk": "공간 리스크",
    "cleanliness_strength": "청결 강점",
    "cleanliness_risk": "청결 리스크",
    "revisit_intent": "재방문 의도",
    "revisit_risk": "재방문 리스크",
    "operation_efficiency": "운영 효율 신호",
}

pd.DataFrame({"label": OPERATION_LABELS, "korean": [LABEL_KO[x] for x in OPERATION_LABELS]})

,label,korean
0,product_strength,제품 경쟁력
1,product_risk,제품 품질 리스크
2,service_strength,서비스 강점
3,service_risk,서비스 리스크
4,price_resistance,가격 저항
5,price_value_strength,가격 대비 만족
6,waiting_risk,대기/혼잡 리스크
7,space_strength,공간 강점
8,space_risk,공간 리스크
9,cleanliness_strength,청결 강점


## 5. 데이터 로드/생성 함수

우선순위는 다음입니다.

1. `data/cafe_review_signal_dataset_v2_multilabel.csv`
2. `data/cafe_review_signal_dataset_v1.csv`
3. 둘 다 없으면 실험용 데이터를 자동 생성

v1.2는 복합 라벨 문장을 더 많이 포함하도록 데이터를 보강합니다.

In [16]:
def normalize_signal(value):
    """operation_signals 값을 항상 list[str] 형태로 안전하게 변환합니다."""
    if isinstance(value, (list, tuple, set, np.ndarray)):
        items = []
        for x in list(value):
            items.extend(normalize_signal(x))
        return sorted(set(items))

    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    if isinstance(value, str):
        text = value.strip()
        if not text or text.lower() in {"nan", "none", "null"}:
            return []
        # 리스트 문자열 처리: "['a', 'b']"
        if text.startswith("[") and text.endswith("]"):
            try:
                parsed = ast.literal_eval(text)
                return normalize_signal(parsed)
            except Exception:
                pass
        # 구분자 처리
        for sep in [";", ",", "|", "/"]:
            text = text.replace(sep, ";")
        return sorted(set([x.strip() for x in text.split(";") if x.strip()]))

    return []


def infer_signals_from_text(text):
    text = str(text)
    signals = []
    rules = [
        ("product_strength", ["맛있", "고소", "진하", "디저트가 좋", "음료가 좋", "메뉴가 좋", "퀄리티", "풍미"]),
        ("product_risk", ["맛없", "싱겁", "탄맛", "별로", "품질", "차갑", "묽", "누락"]),
        ("service_strength", ["친절", "상냥", "응대가 좋", "설명", "배려"]),
        ("service_risk", ["불친절", "응대가 아쉬", "직원이", "무뚝뚝", "불쾌", "대답"]),
        ("price_resistance", ["비싸", "가격이", "가성비가 아쉽", "부담", "양이 적"]),
        ("price_value_strength", ["가성비", "가격 대비", "합리", "구성이 좋"]),
        ("waiting_risk", ["기다", "대기", "오래 걸", "줄이", "혼잡", "붐벼"]),
        ("space_strength", ["분위기", "인테리어", "좌석이 편", "머물기", "공간이 좋", "뷰가"]),
        ("space_risk", ["좁", "시끄", "자리 없", "불편", "답답"]),
        ("cleanliness_strength", ["깔끔", "깨끗", "청결"]),
        ("cleanliness_risk", ["지저분", "테이블 정리", "더럽", "위생", "냄새"]),
        ("revisit_intent", ["다시", "또 오", "재방문", "추천", "방문하고 싶"]),
        ("revisit_risk", ["다시는", "안 갈", "한 번이면", "재방문은", "추천하기 어렵"]),
        ("operation_efficiency", ["빠르게", "회전", "동선", "정리", "처리", "효율"]),
    ]
    for label, kws in rules:
        if any(kw in text for kw in kws):
            signals.append(label)
    return sorted(set(signals))


def base_single_label_templates():
    templates = {
        "product_strength": ["커피 향이 진하고 맛이 좋아요.", "디저트가 신선하고 만족스러웠어요.", "음료 퀄리티가 기대 이상이었어요."],
        "product_risk": ["커피가 너무 묽고 맛이 아쉬웠어요.", "디저트가 오래된 느낌이라 실망했어요.", "음료 맛이 일정하지 않았어요."],
        "service_strength": ["직원분이 친절하게 응대해 주셨어요.", "주문 설명이 자세해서 좋았어요.", "바쁜 시간에도 직원이 상냥했어요."],
        "service_risk": ["직원 응대가 무뚝뚝해서 아쉬웠어요.", "문의했는데 대답이 불친절했어요.", "서비스가 기대보다 부족했어요."],
        "price_resistance": ["가격이 조금 비싸게 느껴졌어요.", "양에 비해 가격 부담이 있었어요.", "가격 대비 만족도가 낮았어요."],
        "price_value_strength": ["가격 대비 구성이 좋아 만족했어요.", "가성비가 좋아 자주 오고 싶어요.", "이 정도 가격이면 충분히 괜찮아요."],
        "waiting_risk": ["주말에는 대기 시간이 너무 길었어요.", "주문 후 음료가 나오기까지 오래 걸렸어요.", "사람이 많아 매장이 혼잡했어요."],
        "space_strength": ["분위기가 좋아 오래 머물기 좋은 카페였어요.", "좌석이 편하고 공간이 아늑했어요.", "인테리어가 예뻐서 사진 찍기 좋았어요."],
        "space_risk": ["좌석 간격이 좁아서 불편했어요.", "매장이 너무 시끄러워 대화하기 어려웠어요.", "자리가 부족해서 오래 머물기 힘들었어요."],
        "cleanliness_strength": ["매장이 깔끔하고 테이블도 깨끗했어요.", "화장실까지 청결해서 좋았어요.", "전체적으로 관리가 잘 된 느낌이었어요."],
        "cleanliness_risk": ["테이블 정리가 늦어서 지저분했어요.", "매장 바닥과 좌석이 깨끗하지 않았어요.", "위생 관리가 조금 아쉬웠어요."],
        "revisit_intent": ["다음에도 다시 방문하고 싶어요.", "친구에게 추천하고 싶은 카페예요.", "근처에 오면 또 들를 것 같아요."],
        "revisit_risk": ["다시는 방문하지 않을 것 같아요.", "한 번 방문으로 충분하다고 느꼈어요.", "재방문 의사는 별로 없어요."],
        "operation_efficiency": ["주문 처리와 픽업 동선이 효율적이었어요.", "바쁜 시간에도 음료가 빠르게 나왔어요.", "직원들의 정리와 운영이 매끄러웠어요."],
    }
    rows = []
    for label, texts in templates.items():
        for i, text in enumerate(texts):
            rows.append({"id": f"base_{label}_{i}", "text": text, "operation_signals": [label]})
    return rows


def composite_templates():
    combos = [
        (["product_strength", "service_risk"], ["커피는 맛있는데 직원 응대가 조금 아쉬웠어요.", "디저트는 좋았지만 직원이 무뚝뚝했어요."]),
        (["price_resistance", "product_strength", "revisit_intent"], ["가격은 조금 비싸지만 디저트가 맛있어서 또 오고 싶어요.", "음료 가격은 부담되지만 맛이 좋아 다시 방문할 것 같아요."]),
        (["waiting_risk", "service_strength"], ["주말에는 오래 기다렸지만 직원분은 친절했어요.", "대기 시간은 길었지만 안내가 친절해서 괜찮았어요."]),
        (["space_strength", "revisit_intent"], ["분위기가 좋아서 오래 머물고 다시 오고 싶어요.", "공간이 아늑해서 다음에도 방문하고 싶어요."]),
        (["product_strength", "cleanliness_risk"], ["커피는 맛있지만 테이블이 지저분해서 아쉬웠어요.", "디저트는 좋았는데 매장 청결은 아쉬웠어요."]),
        (["service_risk", "waiting_risk"], ["직원 응대가 아쉽고 주문도 오래 기다렸어요.", "대기 시간이 길고 서비스도 조금 불친절했어요."]),
        (["price_value_strength", "space_strength", "revisit_intent"], ["가성비도 좋고 분위기도 좋아서 또 오고 싶어요.", "가격 대비 만족스럽고 공간이 좋아 재방문하고 싶어요."]),
        (["space_risk", "product_strength"], ["커피 맛은 좋았지만 좌석이 좁아서 불편했어요.", "음료는 맛있는데 매장이 너무 시끄러웠어요."]),
        (["operation_efficiency", "service_strength"], ["음료가 빠르게 나오고 직원 안내도 친절했어요.", "주문 동선이 편하고 직원 응대도 좋았어요."]),
        (["product_risk", "revisit_risk"], ["음료 맛이 아쉬워서 다시 방문하진 않을 것 같아요.", "디저트 품질이 실망스러워 재방문 의사는 없어요."]),
        (["cleanliness_strength", "space_strength"], ["매장이 깔끔하고 분위기도 좋아서 편하게 머물렀어요.", "공간이 예쁘고 테이블도 깨끗했어요."]),
        (["price_resistance", "space_strength"], ["가격은 비싼 편이지만 분위기는 정말 좋았어요.", "음료 가격은 부담되지만 공간은 만족스러웠어요."]),
    ]
    rows = []
    idx = 0
    for labels, texts in combos:
        for text in texts:
            rows.append({"id": f"composite_{idx:04d}", "text": text, "operation_signals": labels})
            idx += 1
    return rows


def build_synthetic_dataset(repeat=8):
    rows = []
    seed_rows = base_single_label_templates() + composite_templates()
    modifiers = ["", " 전반적으로", " 오늘 방문했는데", " 개인적으로", " 친구와 함께 갔을 때"]
    tails = ["", "라고 느꼈어요.", "라는 생각이 들었습니다.", "점이 기억에 남아요.", "부분이 인상적이었어요."]
    idx = 0
    for r in seed_rows:
        text0 = r["text"].rstrip(".")
        for k in range(repeat):
            prefix = modifiers[k % len(modifiers)]
            tail = tails[k % len(tails)]
            text = f"{prefix} {text0} {tail}".strip()
            rows.append({"id": f"syn_{idx:05d}", "text": re.sub(r"\s+", " ", text), "operation_signals": list(r["operation_signals"])})
            idx += 1
    return pd.DataFrame(rows)


def load_or_create_dataset():
    p2 = DATA_DIR / "cafe_review_signal_dataset_v2_multilabel.csv"
    p1 = DATA_DIR / "cafe_review_signal_dataset_v1.csv"
    if p2.exists():
        print("v2 데이터셋 로드:", p2)
        df = pd.read_csv(p2)
    elif p1.exists():
        print("v1 데이터셋 로드 후 다중 라벨 변환:", p1)
        df = pd.read_csv(p1)
    else:
        print("데이터셋이 없어 실험용 데이터를 생성합니다.")
        df = build_synthetic_dataset(repeat=8)
        df.to_csv(p2, index=False, encoding="utf-8-sig")
        print("생성 저장:", p2)

    if "operation_signals" in df.columns:
        df["operation_signals"] = df["operation_signals"].apply(normalize_signal)
    elif "operation_signal" in df.columns:
        df["operation_signals"] = df["operation_signal"].apply(normalize_signal)
    else:
        df["operation_signals"] = df["text"].apply(infer_signals_from_text)

    df["operation_signals"] = df["operation_signals"].apply(lambda xs: sorted(set([x for x in xs if x in OPERATION_LABELS])))
    df = df[df["operation_signals"].apply(len) > 0].copy()
    df["text"] = df["text"].astype(str).str.strip()
    df = df[df["text"].str.len() > 0].copy()
    return df.reset_index(drop=True)

df = load_or_create_dataset()
print("초기 데이터 수:", len(df))
df.head()

v2 데이터셋 로드: /workspace/data/cafe_review_signal_dataset_v2_multilabel.csv
초기 데이터 수: 528


,id,text,operation_signals
0,syn_00000,커피 향이 진하고 맛이 좋아요,[product_strength]
1,syn_00001,전반적으로 커피 향이 진하고 맛이 좋아요 라고 느꼈어요.,[product_strength]
2,syn_00002,오늘 방문했는데 커피 향이 진하고 맛이 좋아요 라는 생각이 들었습니다.,[product_strength]
3,syn_00003,개인적으로 커피 향이 진하고 맛이 좋아요 점이 기억에 남아요.,[product_strength]
4,syn_00004,친구와 함께 갔을 때 커피 향이 진하고 맛이 좋아요 부분이 인상적이었어요.,[product_strength]


## 6. 중복 제거와 복합 라벨 데이터 보강

v1.1에서 중복이 많이 확인되었으므로 text 기준으로 중복을 제거합니다. 이후 부족한 라벨과 복합 라벨 문장을 보강합니다.

In [17]:
before = len(df)
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
after = len(df)
print(f"중복 제거 전: {before}")
print(f"중복 제거 후: {after}")
print(f"제거된 중복 수: {before - after}")

# 복합 라벨 샘플 추가
composite_df = pd.DataFrame(composite_templates())
# 다양한 표현으로 반복 생성
extra_composites = []
for rep in range(10):
    for _, r in composite_df.iterrows():
        text = r["text"].rstrip(".")
        variants = [
            f"오늘 방문했는데 {text}.",
            f"개인적으로 {text}라고 느꼈어요.",
            f"친구와 함께 갔을 때 {text}.",
            f"전체적으로 {text} 부분이 기억에 남아요.",
        ]
        extra_composites.append({"id": f"extra_comp_{rep}_{_}", "text": variants[rep % len(variants)], "operation_signals": r["operation_signals"]})
extra_df = pd.DataFrame(extra_composites)

# 라벨별 최소 샘플 수 보강
MIN_PER_LABEL = 90
label_counter = Counter()
for labels in df["operation_signals"]:
    label_counter.update(labels)

synthetic_pool = build_synthetic_dataset(repeat=12)
augment_rows = []
for label in OPERATION_LABELS:
    need = max(0, MIN_PER_LABEL - label_counter.get(label, 0))
    if need > 0:
        candidates = synthetic_pool[synthetic_pool["operation_signals"].apply(lambda xs: label in xs)].head(need)
        augment_rows.append(candidates)
        print(f"보강: {label} {need}개")

parts = [df, extra_df]
if augment_rows:
    parts.extend(augment_rows)

df = pd.concat(parts, ignore_index=True)
df["operation_signals"] = df["operation_signals"].apply(normalize_signal)
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

# 라벨이 없는 행 제거
df["operation_signals"] = df["operation_signals"].apply(lambda xs: sorted(set([x for x in xs if x in OPERATION_LABELS])))
df = df[df["operation_signals"].apply(len) > 0].reset_index(drop=True)

# 저장
v12_data_path = DATA_DIR / "cafe_review_signal_dataset_v12_multilabel_augmented.csv"
df.to_csv(v12_data_path, index=False, encoding="utf-8-sig")

label_counter = Counter()
for labels in df["operation_signals"]:
    label_counter.update(labels)

avg_labels = np.mean([len(x) for x in df["operation_signals"]])
print("최종 데이터 수:", len(df))
print("샘플당 평균 라벨 수:", round(avg_labels, 3))
print("저장:", v12_data_path)

pd.DataFrame(label_counter.items(), columns=["label", "count"]).sort_values("count", ascending=False)

중복 제거 전: 528
중복 제거 후: 330
제거된 중복 수: 198
보강: product_strength 35개
보강: product_risk 65개
보강: service_strength 55개
보강: service_risk 55개
보강: price_resistance 55개
보강: price_value_strength 65개
보강: waiting_risk 55개
보강: space_strength 35개
보강: space_risk 65개
보강: cleanliness_strength 65개
보강: cleanliness_risk 65개
보강: revisit_intent 45개
보강: revisit_risk 65개
보강: operation_efficiency 65개
최종 데이터 수: 426
샘플당 평균 라벨 수: 1.592
저장: /workspace/data/cafe_review_signal_dataset_v12_multilabel_augmented.csv


,label,count
0,product_strength,87
7,space_strength,87
11,revisit_intent,69
2,service_strength,51
3,service_risk,51
4,price_resistance,51
6,waiting_risk,51
1,product_risk,33
5,price_value_strength,33
8,space_risk,33


## 7. 다중 라벨 벡터 변환과 분리

다중 라벨 분류에서는 각 라벨을 0/1 벡터로 변환합니다.

In [18]:
mlb = MultiLabelBinarizer(classes=OPERATION_LABELS)
Y = mlb.fit_transform(df["operation_signals"]).astype(np.float32)
label_cols = [f"label_{label}" for label in OPERATION_LABELS]
for i, label in enumerate(OPERATION_LABELS):
    df[f"label_{label}"] = Y[:, i]

print("라벨 벡터 shape:", Y.shape)
print("샘플당 평균 positive label 수:", Y.sum(axis=1).mean())
print("총 positive label 수:", int(Y.sum()))

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, shuffle=True)
valid_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, shuffle=True)

print("train:", len(train_df))
print("valid:", len(valid_df))
print("test:", len(test_df))

def label_summary(split_df, name):
    arr = split_df[label_cols].values
    return pd.DataFrame({
        "split": name,
        "label": OPERATION_LABELS,
        "positive_count": arr.sum(axis=0).astype(int),
    })

split_summary = pd.concat([
    label_summary(train_df, "train"),
    label_summary(valid_df, "valid"),
    label_summary(test_df, "test"),
], ignore_index=True)

split_summary.to_csv(OUTPUT_DIR / "label_distribution_by_split.csv", index=False, encoding="utf-8-sig")
split_summary.pivot(index="label", columns="split", values="positive_count")

라벨 벡터 shape: (426, 14)
샘플당 평균 positive label 수: 1.5915493
총 positive label 수: 678
train: 298
valid: 64
test: 64


split,test,train,valid
label,,,
cleanliness_risk,2,27,4
cleanliness_strength,6,22,5
operation_efficiency,8,19,6
price_resistance,4,38,9
price_value_strength,6,20,7
product_risk,7,19,7
product_strength,8,69,10
revisit_intent,11,50,8
revisit_risk,5,26,2


## 8. HuggingFace Dataset 구성과 Tokenizer 적용

기존 NSMC 실험과 같은 `klue/bert-base`를 사용하되, 문제 유형은 `multi_label_classification`입니다.

In [19]:
MODEL_NAME = "klue/bert-base"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_hf_dataset(split_df):
    tmp = split_df[["text"] + label_cols].copy()
    tmp["labels"] = tmp[label_cols].values.tolist()
    tmp = tmp[["text", "labels"]]
    return Dataset.from_pandas(tmp, preserve_index=False)

raw_datasets = DatasetDict({
    "train": make_hf_dataset(train_df),
    "valid": make_hf_dataset(valid_df),
    "test": make_hf_dataset(test_df),
})

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tokenized_datasets = raw_datasets.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/298 [00:00<?, ? examples/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 298
    })
    valid: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 64
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 64
    })
})

## 9. Weighted Loss 적용 모델과 Trainer

v1.1에서는 모델이 모든 라벨을 낮은 확률로 보는 문제가 있었습니다. 이번에는 `pos_weight`를 사용해 positive label의 학습 비중을 높입니다.

In [20]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(OPERATION_LABELS),
    problem_type="multi_label_classification",
    id2label={i: label for i, label in enumerate(OPERATION_LABELS)},
    label2id={label: i for i, label in enumerate(OPERATION_LABELS)},
)

train_labels = train_df[label_cols].values.astype(np.float32)
pos_counts = train_labels.sum(axis=0)
neg_counts = len(train_labels) - pos_counts
# 너무 큰 가중치는 학습을 불안정하게 만들 수 있어 상한을 둡니다.
pos_weight = np.clip(neg_counts / np.maximum(pos_counts, 1), 1.0, 8.0).astype(np.float32)
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32)

pos_weight_df = pd.DataFrame({
    "label": OPERATION_LABELS,
    "positive_count": pos_counts.astype(int),
    "negative_count": neg_counts.astype(int),
    "pos_weight": pos_weight,
})
pos_weight_df.to_csv(OUTPUT_DIR / "pos_weight_by_label.csv", index=False, encoding="utf-8-sig")
pos_weight_df

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


,label,positive_count,negative_count,pos_weight
0,product_strength,69,229,3.318841
1,product_risk,19,279,8.000000
2,service_strength,34,264,7.764706
3,service_risk,37,261,7.054054
4,price_resistance,38,260,6.842105
5,price_value_strength,20,278,8.000000
6,waiting_risk,38,260,6.842105
7,space_strength,60,238,3.966667
8,space_risk,25,273,8.000000
9,cleanliness_strength,22,276,8.000000


In [21]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

DEFAULT_THRESHOLD = 0.30

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid(logits)
    preds = (probs >= DEFAULT_THRESHOLD).astype(int)
    labels = labels.astype(int)
    empty_rate = (preds.sum(axis=1) == 0).mean()
    return {
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "micro_precision": precision_score(labels, preds, average="micro", zero_division=0),
        "micro_recall": recall_score(labels, preds, average="micro", zero_division=0),
        "empty_prediction_rate": empty_rate,
    }

class WeightedMultilabelTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(logits.device))
        loss = loss_fct(logits, labels.float())
        return (loss, outputs) if return_outputs else loss

## 10. 학습 실행

중간 checkpoint는 저장하지 않습니다. 최종 모델만 별도 저장합니다.

In [22]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "trainer_outputs"),
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy="epoch",
    save_strategy="no",
    load_best_model_at_end=False,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = WeightedMultilabelTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["valid"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    pos_weight=pos_weight_tensor,
)

start_time = time.time()
train_result = trainer.train()
training_time = time.time() - start_time
print(f"학습 시간: {training_time:.2f}초")

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Micro Precision,Micro Recall,Empty Prediction Rate
1,1.132800,0.987576,0.205607,0.204420,0.114716,0.990000,0.000000
2,0.915500,0.777628,0.256082,0.255662,0.146843,1.000000,0.000000
3,0.714000,0.621527,0.316456,0.312313,0.187970,1.000000,0.000000
4,0.577100,0.537158,0.360360,0.355353,0.219780,1.000000,0.000000
5,0.506800,0.489190,0.387597,0.382849,0.240385,1.000000,0.000000
6,0.472000,0.472903,0.397614,0.391311,0.248139,1.000000,0.000000


학습 시간: 9.96초


## 11. 기본 Test 평가

기본 threshold는 0.30으로 둡니다. 하지만 최종 판단은 다음 셀의 threshold 튜닝 결과를 기준으로 합니다.

In [23]:
test_metrics_default = trainer.evaluate(tokenized_datasets["test"])
test_metrics_default["training_time_seconds"] = training_time
with open(OUTPUT_DIR / "test_metrics_default_threshold_030.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics_default, f, ensure_ascii=False, indent=2)
test_metrics_default

{'eval_loss': 0.4742761254310608,
 'eval_micro_f1': 0.39166666666666666,
 'eval_macro_f1': 0.37778799883916686,
 'eval_micro_precision': 0.24352331606217617,
 'eval_micro_recall': 1.0,
 'eval_empty_prediction_rate': 0.0,
 'eval_runtime': 0.052,
 'eval_samples_per_second': 1230.898,
 'eval_steps_per_second': 38.466,
 'epoch': 6.0,
 'training_time_seconds': 9.963258266448975}

## 12. Threshold 튜닝

v1.1의 핵심 문제는 0.5 threshold가 너무 높아 빈 예측이 생긴 것입니다. 여기서는 valid set에서 여러 threshold를 테스트하고 최적값을 찾습니다.

In [24]:
def metrics_at_threshold(y_true, probs, threshold):
    preds = (probs >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "micro_f1": f1_score(y_true, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, preds, average="macro", zero_division=0),
        "micro_precision": precision_score(y_true, preds, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, preds, average="micro", zero_division=0),
        "empty_prediction_rate": float((preds.sum(axis=1) == 0).mean()),
        "avg_predicted_labels": float(preds.sum(axis=1).mean()),
    }

valid_pred = trainer.predict(tokenized_datasets["valid"])
valid_probs = sigmoid(valid_pred.predictions)
valid_labels = np.array(valid_df[label_cols].values).astype(int)

test_pred = trainer.predict(tokenized_datasets["test"])
test_probs = sigmoid(test_pred.predictions)
test_labels = np.array(test_df[label_cols].values).astype(int)

thresholds = np.round(np.arange(0.10, 0.55, 0.05), 2)
valid_threshold_results = pd.DataFrame([metrics_at_threshold(valid_labels, valid_probs, t) for t in thresholds])
valid_threshold_results = valid_threshold_results.sort_values(["micro_f1", "macro_f1"], ascending=False).reset_index(drop=True)

best_threshold = float(valid_threshold_results.iloc[0]["threshold"])
print("best_threshold:", best_threshold)
valid_threshold_results.to_csv(OUTPUT_DIR / "valid_threshold_tuning_results.csv", index=False, encoding="utf-8-sig")
valid_threshold_results

best_threshold: 0.5


,threshold,micro_f1,macro_f1,micro_precision,micro_recall,empty_prediction_rate,avg_predicted_labels
0,0.50,0.769231,0.798040,0.625000,1.0,0.0,2.500000
1,0.45,0.714286,0.742586,0.555556,1.0,0.0,2.812500
2,0.40,0.574713,0.580570,0.403226,1.0,0.0,3.875000
3,0.35,0.501253,0.501461,0.334448,1.0,0.0,4.671875
4,0.30,0.397614,0.391311,0.248139,1.0,0.0,6.296875
5,0.25,0.339559,0.331956,0.204499,1.0,0.0,7.640625
6,0.20,0.269906,0.261963,0.156006,1.0,0.0,10.015625
7,0.15,0.218103,0.213872,0.122399,1.0,0.0,12.765625
8,0.10,0.201207,0.198653,0.111857,1.0,0.0,13.968750


In [25]:
test_threshold_results = pd.DataFrame([metrics_at_threshold(test_labels, test_probs, t) for t in thresholds])
test_threshold_results.to_csv(OUTPUT_DIR / "test_threshold_tuning_results.csv", index=False, encoding="utf-8-sig")

test_best_metrics = metrics_at_threshold(test_labels, test_probs, best_threshold)
with open(OUTPUT_DIR / "test_metrics_best_global_threshold.json", "w", encoding="utf-8") as f:
    json.dump(test_best_metrics, f, ensure_ascii=False, indent=2)

test_threshold_results

,threshold,micro_f1,macro_f1,micro_precision,micro_recall,empty_prediction_rate,avg_predicted_labels
0,0.10,0.189899,0.186529,0.104911,1.0,0.0,14.000000
1,0.15,0.205689,0.199705,0.114634,1.0,0.0,12.812500
2,0.20,0.251337,0.242489,0.143731,1.0,0.0,10.218750
3,0.25,0.319185,0.311335,0.189899,1.0,0.0,7.734375
4,0.30,0.391667,0.377788,0.243523,1.0,0.0,6.031250
5,0.35,0.480818,0.464621,0.316498,1.0,0.0,4.640625
6,0.40,0.566265,0.556807,0.394958,1.0,0.0,3.718750
7,0.45,0.641638,0.630745,0.472362,1.0,0.0,3.109375
8,0.50,0.688645,0.683731,0.525140,1.0,0.0,2.796875


## 13. 라벨별 Threshold 탐색

운영 신호마다 적절한 threshold가 다를 수 있습니다. 예를 들어 `waiting_risk`와 `revisit_intent`의 확률 분포는 다를 수 있습니다.

In [26]:
def best_threshold_for_each_label(y_true, probs, thresholds):
    rows = []
    label_thresholds = {}
    for j, label in enumerate(OPERATION_LABELS):
        best = {"label": label, "threshold": 0.30, "f1": -1, "precision": 0, "recall": 0, "support": int(y_true[:, j].sum())}
        for t in thresholds:
            pred = (probs[:, j] >= t).astype(int)
            f1 = f1_score(y_true[:, j], pred, zero_division=0)
            precision = precision_score(y_true[:, j], pred, zero_division=0)
            recall = recall_score(y_true[:, j], pred, zero_division=0)
            if f1 > best["f1"]:
                best.update({"threshold": float(t), "f1": float(f1), "precision": float(precision), "recall": float(recall)})
        rows.append(best)
        label_thresholds[label] = best["threshold"]
    return pd.DataFrame(rows), label_thresholds

label_threshold_df, label_thresholds = best_threshold_for_each_label(valid_labels, valid_probs, thresholds)
label_threshold_df.to_csv(OUTPUT_DIR / "label_wise_thresholds_valid.csv", index=False, encoding="utf-8-sig")
with open(OUTPUT_DIR / "label_wise_thresholds.json", "w", encoding="utf-8") as f:
    json.dump(label_thresholds, f, ensure_ascii=False, indent=2)
label_threshold_df

,label,threshold,f1,precision,recall,support
0,product_strength,0.50,0.606061,0.434783,1.0,10
1,product_risk,0.50,1.000000,1.000000,1.0,7
2,service_strength,0.40,0.782609,0.642857,1.0,9
3,service_risk,0.50,0.933333,0.875000,1.0,7
4,price_resistance,0.50,0.947368,0.900000,1.0,9
5,price_value_strength,0.45,0.933333,0.875000,1.0,7
6,waiting_risk,0.50,0.583333,0.411765,1.0,7
7,space_strength,0.50,0.866667,0.764706,1.0,13
8,space_risk,0.45,0.631579,0.461538,1.0,6
9,cleanliness_strength,0.50,0.769231,0.625000,1.0,5


In [27]:
def predict_with_label_thresholds(probs, label_thresholds):
    preds = np.zeros_like(probs, dtype=int)
    for j, label in enumerate(OPERATION_LABELS):
        preds[:, j] = (probs[:, j] >= label_thresholds[label]).astype(int)
    return preds

labelwise_preds = predict_with_label_thresholds(test_probs, label_thresholds)
labelwise_metrics = {
    "micro_f1": f1_score(test_labels, labelwise_preds, average="micro", zero_division=0),
    "macro_f1": f1_score(test_labels, labelwise_preds, average="macro", zero_division=0),
    "micro_precision": precision_score(test_labels, labelwise_preds, average="micro", zero_division=0),
    "micro_recall": recall_score(test_labels, labelwise_preds, average="micro", zero_division=0),
    "empty_prediction_rate": float((labelwise_preds.sum(axis=1) == 0).mean()),
    "avg_predicted_labels": float(labelwise_preds.sum(axis=1).mean()),
}
with open(OUTPUT_DIR / "test_metrics_labelwise_threshold.json", "w", encoding="utf-8") as f:
    json.dump(labelwise_metrics, f, ensure_ascii=False, indent=2)
labelwise_metrics

{'micro_f1': 0.6738351254480287,
 'macro_f1': 0.6660701361884467,
 'micro_precision': 0.5081081081081081,
 'micro_recall': 1.0,
 'empty_prediction_rate': 0.0,
 'avg_predicted_labels': 2.890625}

## 14. Top-k 평가

threshold가 낮게 잡히거나 높게 잡히면 서비스 해석이 흔들릴 수 있습니다. 그래서 정답 라벨이 top-k 후보 안에 들어가는지도 확인합니다.

In [28]:
def topk_metrics(y_true, probs, ks=(1,2,3,5)):
    rows = []
    top_order = np.argsort(-probs, axis=1)
    true_sets = [set(np.where(row == 1)[0]) for row in y_true]
    for k in ks:
        hit_any = []
        cover_all = []
        for i in range(len(y_true)):
            pred_set = set(top_order[i, :k])
            true_set = true_sets[i]
            hit_any.append(len(pred_set & true_set) > 0 if true_set else False)
            cover_all.append(true_set.issubset(pred_set) if true_set else False)
        rows.append({
            "top_k": k,
            "hit_any_rate": float(np.mean(hit_any)),
            "cover_all_true_labels_rate": float(np.mean(cover_all)),
        })
    return pd.DataFrame(rows)

topk_df = topk_metrics(test_labels, test_probs, ks=(1,2,3,5))
topk_df.to_csv(OUTPUT_DIR / "test_topk_metrics.csv", index=False, encoding="utf-8-sig")
topk_df

,top_k,hit_any_rate,cover_all_true_labels_rate
0,1,0.859375,0.4375
1,2,0.984375,0.9375
2,3,1.000000,1.0000
3,5,1.000000,1.0000


## 15. 라벨별 성능 리포트

best global threshold와 label-wise threshold 기준의 라벨별 성능을 각각 저장합니다.

In [29]:
best_global_preds = (test_probs >= best_threshold).astype(int)

report_global = classification_report(
    test_labels,
    best_global_preds,
    target_names=OPERATION_LABELS,
    zero_division=0,
    output_dict=True,
)
report_labelwise = classification_report(
    test_labels,
    labelwise_preds,
    target_names=OPERATION_LABELS,
    zero_division=0,
    output_dict=True,
)

report_global_df = pd.DataFrame(report_global).T
report_labelwise_df = pd.DataFrame(report_labelwise).T
report_global_df.to_csv(OUTPUT_DIR / "classification_report_best_global_threshold.csv", encoding="utf-8-sig")
report_labelwise_df.to_csv(OUTPUT_DIR / "classification_report_labelwise_threshold.csv", encoding="utf-8-sig")

report_labelwise_df

,precision,recall,f1-score,support
product_strength,0.500000,1.0,0.666667,8.0
product_risk,0.700000,1.0,0.823529,7.0
service_strength,0.400000,1.0,0.571429,8.0
service_risk,0.777778,1.0,0.875000,7.0
price_resistance,0.500000,1.0,0.666667,4.0
price_value_strength,0.428571,1.0,0.600000,6.0
waiting_risk,0.400000,1.0,0.571429,6.0
space_strength,0.583333,1.0,0.736842,14.0
space_risk,0.222222,1.0,0.363636,2.0
cleanliness_strength,0.375000,1.0,0.545455,6.0


## 16. 오답/미탐지 분석

빈 예측, 놓친 라벨, 과잉 예측을 확인합니다.

In [30]:
def row_label_list(binary_row):
    return [OPERATION_LABELS[j] for j, v in enumerate(binary_row) if v == 1]

analysis_rows = []
test_texts = test_df["text"].tolist()
true_label_lists = test_df["operation_signals"].tolist()

for i, text in enumerate(test_texts):
    true_set = set(true_label_lists[i])
    pred_set = set(row_label_list(labelwise_preds[i]))
    top3_idx = np.argsort(-test_probs[i])[:3]
    top3 = [(OPERATION_LABELS[j], float(test_probs[i, j])) for j in top3_idx]
    analysis_rows.append({
        "text": text,
        "true_labels": ";".join(sorted(true_set)),
        "pred_labels_labelwise_threshold": ";".join(sorted(pred_set)),
        "missed": ";".join(sorted(true_set - pred_set)),
        "extra": ";".join(sorted(pred_set - true_set)),
        "top3": json.dumps(top3, ensure_ascii=False),
        "is_empty_pred": len(pred_set) == 0,
    })

error_df = pd.DataFrame(analysis_rows)
error_df.to_csv(OUTPUT_DIR / "test_prediction_analysis_labelwise_threshold.csv", index=False, encoding="utf-8-sig")
print("빈 예측 비율:", error_df["is_empty_pred"].mean())
error_df.head(20)

빈 예측 비율: 0.0


,text,true_labels,pred_labels_labelwise_threshold,missed,extra,top3,is_empty_pred
0,오늘 방문했는데 가성비도 좋고 분위기도 좋아서 또 오고 싶어요 라는 생각이 들었습니다.,price_value_strength;revisit_intent;space_stre...,price_value_strength;revisit_intent;space_stre...,,,"[[""revisit_intent"", 0.8701565861701965], [""pri...",False
1,개인적으로 디저트는 좋았지만 직원이 무뚝뚝했어요 점이 기억에 남아요.,product_strength;service_risk,product_strength;service_risk,,,"[[""service_risk"", 0.8759993314743042], [""produ...",False
2,가격 대비 만족스럽고 공간이 좋아 재방문하고 싶어요,price_value_strength;revisit_intent;space_stre...,price_value_strength;revisit_intent;space_stre...,,,"[[""revisit_intent"", 0.8745065331459045], [""pri...",False
3,친구와 함께 갔을 때 공간이 아늑해서 다음에도 방문하고 싶어요 부분이 인상적이었어요.,revisit_intent;space_strength,price_value_strength;revisit_intent;space_stre...,,price_value_strength,"[[""revisit_intent"", 0.8290771842002869], [""spa...",False
4,디저트가 신선하고 만족스러웠어요,product_strength,product_strength,,,"[[""product_strength"", 0.658307671546936], [""sp...",False
5,커피 맛은 좋았지만 좌석이 좁아서 불편했어요,product_strength;space_risk,cleanliness_risk;product_strength;space_risk,,cleanliness_risk,"[[""space_risk"", 0.8563323616981506], [""product...",False
6,오늘 방문했는데 바쁜 시간에도 직원이 상냥했어요 라는 생각이 들었습니다.,service_strength,operation_efficiency;service_strength;waiting_...,,operation_efficiency;waiting_risk,"[[""service_strength"", 0.8860265612602234], [""w...",False
7,전반적으로 커피가 너무 묽고 맛이 아쉬웠어요 라고 느꼈어요.,product_risk,cleanliness_risk;product_risk;product_strength...,,cleanliness_risk;product_strength;space_risk,"[[""product_strength"", 0.77167147397995], [""spa...",False
8,개인적으로 공간이 예쁘고 테이블도 깨끗했어요라고 느꼈어요.,cleanliness_strength;space_strength,cleanliness_strength;space_strength,,,"[[""cleanliness_strength"", 0.8613050580024719],...",False
9,전체적으로 관리가 잘 된 느낌이었어요,cleanliness_strength,cleanliness_strength;space_strength,,space_strength,"[[""cleanliness_strength"", 0.831963837146759], ...",False


## 17. 문장/절 단위 분리와 서비스형 예측 함수

복합 리뷰를 절 단위로 나눈 뒤 각 절의 운영 신호를 예측합니다. 서비스에서는 이 방식이 단일 문장 예측보다 자연스럽습니다.

In [31]:
def split_review_clauses(text):
    text = str(text).strip()
    patterns = [
        r"는데\s*", r"지만\s*", r"다만\s*", r"하지만\s*", r"그런데\s*",
        r"그리고\s*", r"또\s*", r"반면\s*", r",", r"\."
    ]
    regex = "|".join(patterns)
    parts = re.split(regex, text)
    parts = [p.strip() for p in parts if len(p.strip()) >= 3]
    return parts if parts else [text]


def predict_multilabel_texts(texts, mode="labelwise", threshold=None, top_k=3, fallback_top_k=2):
    if isinstance(texts, str):
        texts = [texts]

    enc = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
        return_tensors="pt",
    )
    device = model.device
    enc = {k: v.to(device) for k, v in enc.items()}
    model.eval()
    with torch.no_grad():
        logits = model(**enc).logits.detach().cpu().numpy()
    probs = sigmoid(logits)

    results = []
    for text, prob in zip(texts, probs):
        if mode == "labelwise":
            selected = [label for j, label in enumerate(OPERATION_LABELS) if prob[j] >= label_thresholds[label]]
        else:
            th = best_threshold if threshold is None else threshold
            selected = [label for j, label in enumerate(OPERATION_LABELS) if prob[j] >= th]

        top_idx = np.argsort(-prob)[:top_k]
        top_labels = [{"label": OPERATION_LABELS[j], "label_ko": LABEL_KO[OPERATION_LABELS[j]], "score": float(prob[j])} for j in top_idx]

        # 서비스에서는 빈 예측이면 top-k 후보를 fallback으로 사용할 수 있습니다.
        fallback_used = False
        if len(selected) == 0 and fallback_top_k > 0:
            selected = [OPERATION_LABELS[j] for j in np.argsort(-prob)[:fallback_top_k]]
            fallback_used = True

        results.append({
            "text": text,
            "signals": selected,
            "signals_ko": [LABEL_KO[x] for x in selected],
            "top_labels": top_labels,
            "fallback_used": fallback_used,
        })
    return results


def analyze_review_by_clauses(text):
    clauses = split_review_clauses(text)
    clause_results = predict_multilabel_texts(clauses, mode="labelwise", top_k=3, fallback_top_k=1)
    merged = sorted(set([sig for r in clause_results for sig in r["signals"]]))
    return {
        "original_text": text,
        "clauses": clause_results,
        "merged_signals": merged,
        "merged_signals_ko": [LABEL_KO[x] for x in merged],
    }

sample_reviews = [
    "커피는 맛있는데 직원 응대가 조금 아쉬웠어요.",
    "가격은 조금 비싸지만 디저트가 맛있어서 또 오고 싶어요.",
    "주말에는 사람이 너무 많아서 주문까지 오래 기다렸지만 직원은 친절했어요.",
    "분위기는 좋은데 테이블 정리가 늦어서 지저분하게 느껴졌어요.",
]

sample_results = [analyze_review_by_clauses(x) for x in sample_reviews]
print(json.dumps(sample_results, ensure_ascii=False, indent=2))
with open(OUTPUT_DIR / "sample_clause_predictions.json", "w", encoding="utf-8") as f:
    json.dump(sample_results, f, ensure_ascii=False, indent=2)

[
  {
    "original_text": "커피는 맛있는데 직원 응대가 조금 아쉬웠어요.",
    "clauses": [
      {
        "text": "커피는 맛있",
        "signals": [
          "product_strength",
          "product_risk",
          "space_risk"
        ],
        "signals_ko": [
          "제품 경쟁력",
          "제품 품질 리스크",
          "공간 리스크"
        ],
        "top_labels": [
          {
            "label": "product_strength",
            "label_ko": "제품 경쟁력",
            "score": 0.7074047327041626
          },
          {
            "label": "space_risk",
            "label_ko": "공간 리스크",
            "score": 0.6245225667953491
          },
          {
            "label": "product_risk",
            "label_ko": "제품 품질 리스크",
            "score": 0.5246534943580627
          }
        ],
        "fallback_used": false
      },
      {
        "text": "직원 응대가 조금 아쉬웠어요",
        "signals": [
          "service_strength",
          "service_risk",
          "waiting_risk"
        ],
        "signals_ko": [
          "서비스 강점"

## 18. Store DNA 점수화

예측된 운영 신호를 매장 DNA 점수로 변환합니다. 실제 서비스에서는 POS 데이터와 결합해 점수를 보정해야 합니다.

In [32]:
DNA_MAPPING = {
    "product_power": {"name": "제품 경쟁력", "positive": ["product_strength"], "negative": ["product_risk"]},
    "service_stability": {"name": "서비스 안정성", "positive": ["service_strength"], "negative": ["service_risk"]},
    "price_acceptance": {"name": "가격 수용성", "positive": ["price_value_strength"], "negative": ["price_resistance"]},
    "space_attractiveness": {"name": "공간 매력도", "positive": ["space_strength", "cleanliness_strength"], "negative": ["space_risk", "cleanliness_risk"]},
    "waiting_stability": {"name": "대기/혼잡 안정성", "positive": ["operation_efficiency"], "negative": ["waiting_risk"]},
    "revisit_power": {"name": "재방문 신호", "positive": ["revisit_intent"], "negative": ["revisit_risk"]},
}


def compute_store_dna_from_signals(signal_lists):
    counts = Counter()
    for signals in signal_lists:
        counts.update(signals)
    total = max(sum(counts.values()), 1)

    rows = []
    for key, info in DNA_MAPPING.items():
        pos = sum(counts[x] for x in info["positive"])
        neg = sum(counts[x] for x in info["negative"])
        raw = 50 + 50 * ((pos - neg) / max(pos + neg, 1))
        # 신호량이 너무 적으면 50점에 가까워지도록 보수적으로 조정
        volume = min((pos + neg) / max(total * 0.3, 1), 1)
        score = 50 + (raw - 50) * volume
        rows.append({
            "항목": info["name"],
            "점수": round(float(score), 1),
            "긍정신호수": int(pos),
            "부정신호수": int(neg),
        })
    return pd.DataFrame(rows).sort_values("점수", ascending=False).reset_index(drop=True)

# 샘플 예측 결과 기반 DNA 점수
sample_signal_lists = [r["merged_signals"] for r in sample_results]
dna_df = compute_store_dna_from_signals(sample_signal_lists)
dna_df.to_csv(OUTPUT_DIR / "sample_store_dna_scores.csv", index=False, encoding="utf-8-sig")
dna_df

,항목,점수,긍정신호수,부정신호수
0,제품 경쟁력,68.5,3,1
1,서비스 안정성,59.3,2,1
2,재방문 신호,59.3,1,0
3,가격 수용성,50.0,1,1
4,대기/혼잡 안정성,40.7,1,2
5,공간 매력도,3.7,0,5


## 19. 사장님용 해석 문장 예시

모델 출력은 최종 리포트가 아니라, LLM 또는 규칙 기반 리포트 생성기의 입력입니다.

In [33]:
def generate_simple_store_comment(dna_df):
    strongest = dna_df.sort_values("점수", ascending=False).head(2)
    weakest = dna_df.sort_values("점수").head(2)
    strong_text = ", ".join([f"{row['항목']}({int(row['점수'])}점)" for _, row in strongest.iterrows()])
    weak_text = ", ".join([f"{row['항목']}({int(row['점수'])}점)" for _, row in weakest.iterrows()])
    return (
        f"이번 리뷰 신호를 보면 강점은 {strong_text}입니다. "
        f"반대로 우선 점검해야 할 항목은 {weak_text}입니다. "
        "다음 단계에서는 이 신호를 POS의 시간대별 매출, 메뉴별 매출, 피크타임 주문량과 결합해 실제 운영 병목을 확인해야 합니다."
    )

comment = generate_simple_store_comment(dna_df)
print(comment)
with open(OUTPUT_DIR / "sample_store_comment.txt", "w", encoding="utf-8") as f:
    f.write(comment)

이번 리뷰 신호를 보면 강점은 제품 경쟁력(68점), 서비스 안정성(59점)입니다. 반대로 우선 점검해야 할 항목은 공간 매력도(3점), 대기/혼잡 안정성(40점)입니다. 다음 단계에서는 이 신호를 POS의 시간대별 매출, 메뉴별 매출, 피크타임 주문량과 결합해 실제 운영 병목을 확인해야 합니다.


## 20. 최종 모델/tokenizer 저장 및 산출물 검증

이 셀을 실행하면 서비스 추론에 필요한 HuggingFace 모델 폴더가 생성됩니다.

생성 목표 파일:

```text
config.json
model.safetensors 또는 pytorch_model.bin
tokenizer.json
tokenizer_config.json
special_tokens_map.json
metadata.json
label2id.json
id2label.json
operation_labels.json
label_thresholds.json
```

> 중요: 이 셀은 학습이 끝난 뒤 실행해야 합니다. 즉, `model`, `tokenizer`, `best_threshold`, `label_thresholds`가 준비된 상태에서 실행합니다.


In [34]:
# ============================================================
# 20. 최종 모델/tokenizer 저장 및 산출물 검증
# ============================================================
# 목적:
# - 학습 완료된 Transformer 모델을 서비스에서 다시 불러올 수 있는 형태로 저장합니다.
# - config.json, model.safetensors 또는 pytorch_model.bin, tokenizer.json,
#   tokenizer_config.json, special_tokens_map.json 생성 여부를 검증합니다.

from pathlib import Path
import json
import shutil

# 1) 필수 객체 확인
required_runtime_objects = ["model", "tokenizer", "OPERATION_LABELS", "LABEL_KO"]
missing_runtime_objects = [name for name in required_runtime_objects if name not in globals()]
if missing_runtime_objects:
    raise NameError(
        "모델 저장에 필요한 객체가 없습니다: " + ", ".join(missing_runtime_objects) + "\n"
        "앞의 학습 셀을 먼저 순서대로 실행하세요."
    )

# 2) 저장 경로 고정
# 기존 MODEL_DIR이 있으면 사용하고, 없으면 기본 경로를 생성합니다.
BASE_DIR = globals().get("BASE_DIR", Path.cwd())
MODEL_DIR = globals().get(
    "MODEL_DIR",
    BASE_DIR / "models" / "cafe_review_signal_multilabel_v1_2"
)
MODEL_DIR = Path(MODEL_DIR)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 완전한 최종본을 위해 이전 저장물을 지우고 다시 저장합니다.
# 같은 폴더를 계속 쓰면 오래된 파일이 남아 혼동될 수 있습니다.
if MODEL_DIR.exists():
    shutil.rmtree(MODEL_DIR)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 3) 라벨 mapping을 config에도 명시적으로 반영
label2id = {label: i for i, label in enumerate(OPERATION_LABELS)}
id2label = {i: label for i, label in enumerate(OPERATION_LABELS)}

model.config.label2id = label2id
model.config.id2label = id2label
model.config.problem_type = "multi_label_classification"
model.config.num_labels = len(OPERATION_LABELS)

# 4) threshold 값 안전 처리
# best_threshold / label_thresholds가 없으면 기본 threshold로 대체합니다.
DEFAULT_THRESHOLD = globals().get("DEFAULT_THRESHOLD", 0.5)
best_threshold_to_save = globals().get("best_threshold", DEFAULT_THRESHOLD)

if "label_thresholds" in globals() and isinstance(label_thresholds, dict):
    label_thresholds_to_save = {label: float(label_thresholds.get(label, best_threshold_to_save)) for label in OPERATION_LABELS}
else:
    label_thresholds_to_save = {label: float(best_threshold_to_save) for label in OPERATION_LABELS}

MAX_LENGTH = globals().get("MAX_LENGTH", 128)
MODEL_NAME = globals().get("MODEL_NAME", "unknown")

# 5) 모델 가중치 저장
# safe_serialization=True이면 model.safetensors가 생성됩니다.
# 환경에 따라 safetensors 저장이 실패하면 pytorch_model.bin으로 자동 대체합니다.
try:
    model.save_pretrained(str(MODEL_DIR), safe_serialization=True)
    weight_save_type = "model.safetensors"
except Exception as e:
    print("safe_serialization 저장 실패. pytorch_model.bin 방식으로 재시도합니다.")
    print("원인:", repr(e))
    model.save_pretrained(str(MODEL_DIR), safe_serialization=False)
    weight_save_type = "pytorch_model.bin"

# 6) tokenizer 저장
# Fast tokenizer인 경우 tokenizer.json이 함께 생성됩니다.
tokenizer.save_pretrained(str(MODEL_DIR))

# tokenizer.json이 생성되지 않은 경우, fast tokenizer backend가 있으면 직접 저장을 시도합니다.
tokenizer_json_path = MODEL_DIR / "tokenizer.json"
if not tokenizer_json_path.exists() and getattr(tokenizer, "is_fast", False):
    try:
        tokenizer.backend_tokenizer.save(str(tokenizer_json_path))
        print("tokenizer.json 직접 저장 완료")
    except Exception as e:
        print("tokenizer.json 직접 저장 실패:", repr(e))

# 7) 서비스 추론용 메타데이터 저장
metadata = {
    "model_name": MODEL_NAME,
    "task": "cafe_review_signal_multilabel_classification_v1_2",
    "model_dir": str(MODEL_DIR),
    "labels": list(OPERATION_LABELS),
    "label_ko": LABEL_KO,
    "label2id": label2id,
    "id2label": {str(k): v for k, v in id2label.items()},
    "best_global_threshold": float(best_threshold_to_save),
    "label_wise_thresholds": label_thresholds_to_save,
    "default_threshold": float(DEFAULT_THRESHOLD),
    "max_length": int(MAX_LENGTH),
    "weight_file_type": weight_save_type,
    "notes": (
        "v1.2 multilabel cafe review signal model. "
        "Use metadata.json and label_thresholds.json for service inference."
    ),
}

with open(MODEL_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

with open(MODEL_DIR / "label2id.json", "w", encoding="utf-8") as f:
    json.dump(label2id, f, ensure_ascii=False, indent=2)

with open(MODEL_DIR / "id2label.json", "w", encoding="utf-8") as f:
    json.dump({str(k): v for k, v in id2label.items()}, f, ensure_ascii=False, indent=2)

with open(MODEL_DIR / "operation_labels.json", "w", encoding="utf-8") as f:
    json.dump(list(OPERATION_LABELS), f, ensure_ascii=False, indent=2)

with open(MODEL_DIR / "label_thresholds.json", "w", encoding="utf-8") as f:
    json.dump(label_thresholds_to_save, f, ensure_ascii=False, indent=2)

# 8) 필수 파일 검증
files = sorted([p.name for p in MODEL_DIR.iterdir() if p.is_file()])

required_exact_files = [
    "config.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "metadata.json",
    "label2id.json",
    "id2label.json",
    "operation_labels.json",
    "label_thresholds.json",
]

missing_exact_files = [name for name in required_exact_files if not (MODEL_DIR / name).exists()]

has_weight_file = (MODEL_DIR / "model.safetensors").exists() or (MODEL_DIR / "pytorch_model.bin").exists()
has_tokenizer_json = (MODEL_DIR / "tokenizer.json").exists()

print("모델 저장 경로:", MODEL_DIR)
print("가중치 저장 방식:", weight_save_type)
print("저장된 파일 목록:")
for name in files:
    print(" -", name)

if missing_exact_files:
    print("\n누락된 필수 파일:", missing_exact_files)

if not has_weight_file:
    raise FileNotFoundError("model.safetensors 또는 pytorch_model.bin 파일이 생성되지 않았습니다.")

if not has_tokenizer_json:
    raise FileNotFoundError(
        "tokenizer.json 파일이 생성되지 않았습니다. "
        "AutoTokenizer가 fast tokenizer로 로드되었는지 확인하세요."
    )

if missing_exact_files:
    raise FileNotFoundError("일부 필수 파일이 누락되었습니다: " + ", ".join(missing_exact_files))

print("\n저장 검증 완료")
print("서비스 추론에 필요한 모델/tokenizer 파일이 준비되었습니다.")


모델 저장 경로: /workspace/models/cafe_review_signal_multilabel_v1_2
가중치 저장 방식: model.safetensors
저장된 파일 목록:
 - config.json
 - id2label.json
 - label2id.json
 - label_thresholds.json
 - metadata.json
 - model.safetensors
 - operation_labels.json
 - special_tokens_map.json
 - tokenizer.json
 - tokenizer_config.json
 - vocab.txt

저장 검증 완료
서비스 추론에 필요한 모델/tokenizer 파일이 준비되었습니다.


## 20-1. 저장된 모델 재로드 테스트

저장된 폴더가 실제 서비스 코드에서 다시 로드되는지 확인합니다. 이 셀이 성공하면 `AutoTokenizer.from_pretrained(MODEL_DIR)`와 `AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)` 방식으로 서비스에서 사용할 수 있습니다.


In [35]:
# ============================================================
# 20-1. 저장된 모델 재로드 테스트
# ============================================================
from transformers import AutoTokenizer, AutoModelForSequenceClassification

reloaded_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
reloaded_model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR))

print("재로드 성공")
print("모델 라벨 수:", reloaded_model.config.num_labels)
print("tokenizer fast 여부:", getattr(reloaded_tokenizer, "is_fast", None))
print("config problem_type:", reloaded_model.config.problem_type)
print("예시 id2label:", reloaded_model.config.id2label)


재로드 성공
모델 라벨 수: 14
tokenizer fast 여부: True
config problem_type: multi_label_classification
예시 id2label: {0: 'product_strength', 1: 'product_risk', 2: 'service_strength', 3: 'service_risk', 4: 'price_resistance', 5: 'price_value_strength', 6: 'waiting_risk', 7: 'space_strength', 8: 'space_risk', 9: 'cleanliness_strength', 10: 'cleanliness_risk', 11: 'revisit_intent', 12: 'revisit_risk', 13: 'operation_efficiency'}


## 21. FastAPI 연결용 predictor 초안 저장

서비스에서는 `predict_review_signals()` 함수를 API 안에서 호출하면 됩니다.

In [36]:
SERVICE_PREDICTOR_CODE = r'''
from pathlib import Path
import re
import json
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def split_review_clauses(text):
    text = str(text).strip()
    patterns = [r"는데\s*", r"지만\s*", r"다만\s*", r"하지만\s*", r"그런데\s*", r"그리고\s*", r"또\s*", r"반면\s*", r",", r"\."]
    parts = re.split("|".join(patterns), text)
    parts = [p.strip() for p in parts if len(p.strip()) >= 3]
    return parts if parts else [text]

class CafeReviewSignalPredictor:
    def __init__(self, model_dir):
        self.model_dir = Path(model_dir)
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(self.model_dir)
        self.model.eval()
        with open(self.model_dir / "metadata.json", "r", encoding="utf-8") as f:
            self.meta = json.load(f)
        self.labels = self.meta["labels"]
        self.label_ko = self.meta["label_ko"]
        self.label_thresholds = self.meta["label_wise_thresholds"]
        self.max_length = self.meta.get("max_length", 128)

    def predict_texts(self, texts, top_k=3, fallback_top_k=1):
        if isinstance(texts, str):
            texts = [texts]
        enc = self.tokenizer(texts, truncation=True, max_length=self.max_length, padding=True, return_tensors="pt")
        with torch.no_grad():
            logits = self.model(**enc).logits.detach().cpu().numpy()
        probs = sigmoid(logits)
        results = []
        for text, prob in zip(texts, probs):
            selected = [label for j, label in enumerate(self.labels) if prob[j] >= self.label_thresholds[label]]
            fallback_used = False
            if not selected and fallback_top_k > 0:
                selected = [self.labels[j] for j in np.argsort(-prob)[:fallback_top_k]]
                fallback_used = True
            top_idx = np.argsort(-prob)[:top_k]
            results.append({
                "text": text,
                "signals": selected,
                "signals_ko": [self.label_ko[x] for x in selected],
                "top_labels": [{"label": self.labels[j], "label_ko": self.label_ko[self.labels[j]], "score": float(prob[j])} for j in top_idx],
                "fallback_used": fallback_used,
            })
        return results

    def predict_review_signals(self, text):
        clauses = split_review_clauses(text)
        clause_results = self.predict_texts(clauses)
        merged = sorted(set(sig for r in clause_results for sig in r["signals"]))
        return {
            "original_text": text,
            "clauses": clause_results,
            "merged_signals": merged,
            "merged_signals_ko": [self.label_ko[x] for x in merged],
        }
'''

predictor_path = SERVICE_DIR / "review_signal_predictor_v1_2.py"
with open(predictor_path, "w", encoding="utf-8") as f:
    f.write(SERVICE_PREDICTOR_CODE)
print("저장 완료:", predictor_path)

저장 완료: /workspace/app/review_signal_predictor_v1_2.py


## 22. 이번 v1.2에서 확인해야 할 결론

v1.2의 성공 기준은 다음입니다.

1. `empty_prediction_rate`가 v1.1보다 낮아졌는가?
2. threshold 튜닝 후 `micro_f1`, `macro_f1`이 0에서 벗어났는가?
3. top-2 또는 top-3 후보 안에 정답 라벨이 들어가는가?
4. 복합 문장에서 2개 이상의 운영 신호가 추출되는가?
5. Store DNA 점수화가 서비스 해석으로 연결되는가?

이 기준을 통과하면 다음 단계는 실제 카페 리뷰 데이터 1,000개 이상 수집과 사람 검수 라벨링입니다.

## 23. 다음 단계

1. 실제 카페 리뷰 1,000개 이상 수집
2. 사람이 검수한 다중 라벨 데이터셋 구축
3. 라벨별 최소 100개 이상 확보
4. 복합 라벨 샘플 비율을 40% 이상으로 구성
5. threshold/label-wise threshold를 실제 validation set 기준으로 재탐색
6. POS 데이터와 결합해 Store DNA 점수 보정
7. FastAPI 예측 API 연결
8. 장사비서 리포트 생성 엔진에 통합

> 핵심: v1.2는 서비스 모델 완성이 아니라, “다중 라벨 예측이 서비스형 구조로 작동하는지”를 확인하는 개선 실험입니다.

## 24. 최종 사용 방법

1. 이 노트북을 위에서 아래로 순서대로 실행합니다.
2. 학습 완료 후 `20. 최종 모델/tokenizer 저장 및 산출물 검증` 셀을 실행합니다.
3. 아래 폴더가 생성되었는지 확인합니다.

```text
models/cafe_review_signal_multilabel_v1_2/
```

4. 이 폴더 안에 다음 파일이 있어야 합니다.

```text
config.json
model.safetensors 또는 pytorch_model.bin
tokenizer.json
tokenizer_config.json
special_tokens_map.json
metadata.json
label2id.json
id2label.json
operation_labels.json
label_thresholds.json
```

5. 이 폴더를 FastAPI 또는 추론 파이프라인에서 그대로 불러오면 됩니다.



# 25. 저장된 모델로 실제 카페 리뷰 1,000개 분석하기

이 섹션은 서비스 추론 파이프라인입니다.

앞쪽 학습 과정에서 저장한 모델을 다시 로드한 뒤,  
`all_cafe_reviews_1000.csv`를 분석하여 다음 파일을 생성합니다.

```text
outputs/real_reviews_model_predictions.csv
outputs/real_reviews_signal_summary.csv
outputs/real_reviews_store_dna_report.json
outputs/real_reviews_owner_report.txt
```

이 부분은 나중에 FastAPI 서비스의 `/analyze-reviews` API로 분리할 수 있습니다.


In [37]:

# ============================================================
# 25-1. 저장 모델 + 실제 리뷰 CSV 자동 탐색
# ============================================================

from pathlib import Path
import json, re, os, math
from collections import Counter

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 모델 폴더 후보
MODEL_DIR_CANDIDATES = [
    PROJECT_ROOT / "models" / "cafe_review_signal_multilabel_v1_2",
    PROJECT_ROOT / "models" / "cafe_review_multilabel_v1_2",
    PROJECT_ROOT / "cafe_review_signal_multilabel_v1_2",
    Path("/workspace/models/cafe_review_signal_multilabel_v1_2"),
    Path("/workspace/models/cafe_review_multilabel_v1_2"),
    Path("/mnt/data/models/cafe_review_signal_multilabel_v1_2"),
]

MODEL_DIR = None
for p in MODEL_DIR_CANDIDATES:
    if p.exists() and (p / "config.json").exists():
        MODEL_DIR = p
        break

if MODEL_DIR is None:
    raise FileNotFoundError(
        "저장된 모델 폴더를 찾지 못했습니다. 먼저 모델 저장 셀을 실행하세요.\n"
        + "\n".join([str(p) for p in MODEL_DIR_CANDIDATES])
    )

# CSV 후보
CSV_CANDIDATES = [
    PROJECT_ROOT / "all_cafe_reviews_1000.csv",
    PROJECT_ROOT / "data" / "all_cafe_reviews_1000.csv",
    Path("/workspace/all_cafe_reviews_1000.csv"),
    Path("/mnt/data/all_cafe_reviews_1000.csv"),
    Path("/mnt/data/all_cafe_reviews_1000(1).csv"),
]

CSV_PATH = None
for p in CSV_CANDIDATES:
    if p.exists():
        CSV_PATH = p
        break

if CSV_PATH is None:
    raise FileNotFoundError(
        "all_cafe_reviews_1000.csv 파일을 찾지 못했습니다.\n"
        + "\n".join([str(p) for p in CSV_CANDIDATES])
    )

print("MODEL_DIR:", MODEL_DIR)
print("CSV_PATH:", CSV_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)


MODEL_DIR: /workspace/models/cafe_review_signal_multilabel_v1_2
CSV_PATH: /workspace/all_cafe_reviews_1000.csv
OUTPUT_DIR: /workspace/outputs


In [38]:

# ============================================================
# 25-2. 저장된 모델/tokenizer 로드
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 저장된 operation_labels / threshold 로드
labels_path = MODEL_DIR / "operation_labels.json"
thresholds_path = MODEL_DIR / "label_thresholds.json"

if labels_path.exists():
    with open(labels_path, "r", encoding="utf-8") as f:
        OPERATION_LABELS = json.load(f)
else:
    OPERATION_LABELS = [
        "product_strength", "product_risk",
        "service_strength", "service_risk",
        "price_value_strength", "price_resistance",
        "waiting_risk",
        "space_strength", "space_risk",
        "cleanliness_strength", "cleanliness_risk",
        "revisit_intent", "revisit_risk",
        "operation_efficiency",
    ]

if thresholds_path.exists():
    with open(thresholds_path, "r", encoding="utf-8") as f:
        LABEL_THRESHOLDS = json.load(f)
else:
    LABEL_THRESHOLDS = {label: 0.5 for label in OPERATION_LABELS}

LABEL_KO = {
    "product_strength": "제품/메뉴 강점",
    "product_risk": "제품/메뉴 리스크",
    "service_strength": "서비스 강점",
    "service_risk": "서비스 리스크",
    "price_value_strength": "가격 대비 만족",
    "price_resistance": "가격 저항",
    "waiting_risk": "대기/혼잡 리스크",
    "space_strength": "공간/분위기 강점",
    "space_risk": "공간/좌석 리스크",
    "cleanliness_strength": "청결 강점",
    "cleanliness_risk": "청결 리스크",
    "revisit_intent": "재방문 의도",
    "revisit_risk": "재방문 저하",
    "operation_efficiency": "운영 효율",
}

print("모델 로드 완료")
print("device:", device)
print("라벨 수:", len(OPERATION_LABELS))
print("가중치 파일 존재:", (MODEL_DIR / "model.safetensors").exists() or (MODEL_DIR / "pytorch_model.bin").exists())


모델 로드 완료
device: cuda
라벨 수: 14
가중치 파일 존재: True


In [39]:

# ============================================================
# 25-3. 실제 리뷰 CSV 로드 및 review_text 생성
# ============================================================

def clean_text(text):
    text = "" if pd.isna(text) else str(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df_real = pd.read_csv(CSV_PATH)

print("원본 크기:", df_real.shape)
print("컬럼:", df_real.columns.tolist())

# title + snippet이 있으면 우선 사용
if "title" in df_real.columns:
    df_real["title_clean"] = df_real["title"].apply(clean_text)
else:
    df_real["title_clean"] = ""

if "snippet" in df_real.columns:
    df_real["snippet_clean"] = df_real["snippet"].apply(clean_text)
else:
    df_real["snippet_clean"] = ""

fallback_text = df_real.select_dtypes(include=["object"]).fillna("").astype(str).agg(" ".join, axis=1)
df_real["review_text"] = (df_real["title_clean"] + " " + df_real["snippet_clean"]).str.strip()

empty_mask = df_real["review_text"].str.len() == 0
df_real.loc[empty_mask, "review_text"] = fallback_text[empty_mask].apply(clean_text)

before = len(df_real)
df_real = df_real[df_real["review_text"].str.len() > 0].copy()
df_real = df_real.drop_duplicates(subset=["review_text"]).reset_index(drop=True)
after = len(df_real)

print("전처리 전:", before)
print("전처리 후:", after)
print("제거:", before - after)

display(df_real.head())


원본 크기: (1000, 9)
컬럼: ['area', 'source', 'query', 'title', 'snippet', 'url', 'domain', 'postdate', 'collected_at']
전처리 전: 1000
전처리 후: 1000
제거: 0


/tmp/ipykernel_1613/264507620.py:27: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  fallback_text = df_real.select_dtypes(include=["object"]).fillna("").astype(str).agg(" ".join, axis=1)


,area,source,query,title,snippet,url,domain,postdate,collected_at,title_clean,snippet_clean,review_text
0,압구정,naver_blog_search,압구정 카페 후기,압구정로데오카페 부캐드뺑 내돈내산 솔직후기 상하이모찌...,솔직한 후기를 한번 남겨보도록 할게요 ! 빵사진도 모두 찍어왔으니 구경해보고 가세요...,https://blog.naver.com/rezel_/224260161959,blog.naver.com,20260421,2026-05-21T13:56:24+09:00,압구정로데오카페 부캐드뺑 내돈내산 솔직후기 상하이모찌...,솔직한 후기를 한번 남겨보도록 할게요 ! 빵사진도 모두 찍어왔으니 구경해보고 가세요...,압구정로데오카페 부캐드뺑 내돈내산 솔직후기 상하이모찌... 솔직한 후기를 한번 남겨...
1,압구정,naver_blog_search,압구정 카페 후기,메종드구르메 우베라떼 압구정로데오 카페 추천 후기,메종드구르메 우베라떼 압구정로데오 카페 추천 후기 평소 분위기 좋은 카페 다니는 걸...,https://blog.naver.com/mare8099/224283391789,blog.naver.com,20260512,2026-05-21T13:56:24+09:00,메종드구르메 우베라떼 압구정로데오 카페 추천 후기,메종드구르메 우베라떼 압구정로데오 카페 추천 후기 평소 분위기 좋은 카페 다니는 걸...,메종드구르메 우베라떼 압구정로데오 카페 추천 후기 메종드구르메 우베라떼 압구정로데오...
2,압구정,naver_blog_search,압구정 카페 후기,압구정 신상카페 안목 갤러리 나팔 NAFAL 경매 첫 방문 후기,압구정신상카페 안목 갤러리에서 나팔 NAFAL 오프라인 경매 관람도 해보고 미술 작...,https://blog.naver.com/ko_jaeeun/224276650630,blog.naver.com,20260506,2026-05-21T13:56:24+09:00,압구정 신상카페 안목 갤러리 나팔 NAFAL 경매 첫 방문 후기,압구정신상카페 안목 갤러리에서 나팔 NAFAL 오프라인 경매 관람도 해보고 미술 작...,압구정 신상카페 안목 갤러리 나팔 NAFAL 경매 첫 방문 후기 압구정신상카페 안목...
3,압구정,naver_blog_search,압구정 카페 후기,추천 압구정 이웃집통통이 버터떡 압구정로데오 카페 후기,메뉴판 압구정 카페 메뉴판인데요. 놀랍게도 매장 이용은 베이커리를 포함해서 1인 1...,https://blog.naver.com/oyssoa/224208658017,blog.naver.com,20260308,2026-05-21T13:56:24+09:00,추천 압구정 이웃집통통이 버터떡 압구정로데오 카페 후기,메뉴판 압구정 카페 메뉴판인데요. 놀랍게도 매장 이용은 베이커리를 포함해서 1인 1...,추천 압구정 이웃집통통이 버터떡 압구정로데오 카페 후기 메뉴판 압구정 카페 메뉴판인...
4,압구정,naver_blog_search,압구정 카페 후기,압구정 베이커리 카페 서울 3대 크루아상 플링크 솔직후기,히히 압구정 베이커리 카페 서울 3대 크루아상 플링크 솔직후기 내부는 만석이라 촬영...,https://blog.naver.com/nanoom23/224288802230,blog.naver.com,20260518,2026-05-21T13:56:24+09:00,압구정 베이커리 카페 서울 3대 크루아상 플링크 솔직후기,히히 압구정 베이커리 카페 서울 3대 크루아상 플링크 솔직후기 내부는 만석이라 촬영...,압구정 베이커리 카페 서울 3대 크루아상 플링크 솔직후기 히히 압구정 베이커리 카페...


In [40]:

# ============================================================
# 25-4. 실제 리뷰 1,000개 모델 예측
# ============================================================

def sigmoid_np(x):
    return 1 / (1 + np.exp(-x))

def predict_texts(texts, batch_size=32, max_length=160, fallback_top_k=2):
    all_probs = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        inputs = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt",
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            logits = model(**inputs).logits.detach().cpu().numpy()

        probs = sigmoid_np(logits)
        all_probs.append(probs)

    all_probs = np.vstack(all_probs)
    pred_bin = np.zeros_like(all_probs, dtype=int)

    for j, label in enumerate(OPERATION_LABELS):
        threshold = float(LABEL_THRESHOLDS.get(label, 0.5))
        pred_bin[:, j] = (all_probs[:, j] >= threshold).astype(int)

    # 빈 예측 fallback: top-k 라벨을 후보로 부여
    for i in range(len(pred_bin)):
        if pred_bin[i].sum() == 0:
            top_idx = np.argsort(-all_probs[i])[:fallback_top_k]
            pred_bin[i, top_idx] = 1

    results = []
    for i in range(len(texts)):
        labels = [OPERATION_LABELS[j] for j, v in enumerate(pred_bin[i]) if v == 1]
        labels_ko = [LABEL_KO.get(label, label) for label in labels]
        top3_idx = np.argsort(-all_probs[i])[:3]
        top3 = [
            {
                "label": OPERATION_LABELS[j],
                "label_ko": LABEL_KO.get(OPERATION_LABELS[j], OPERATION_LABELS[j]),
                "score": float(all_probs[i, j]),
            }
            for j in top3_idx
        ]
        max_conf = float(all_probs[i].max())
        results.append({
            "predicted_signals": labels,
            "predicted_signals_ko": labels_ko,
            "top3_signals": top3,
            "max_confidence": max_conf,
            "num_predicted_labels": len(labels),
            "needs_human_review": bool(
                max_conf < 0.55
                or len(labels) >= 4
                or any(x in labels for x in ["price_resistance", "service_risk", "cleanliness_risk"])
            ),
        })

    return results

texts = df_real["review_text"].astype(str).tolist()
pred_results = predict_texts(texts, batch_size=32)

df_pred = df_real.copy()
df_pred["predicted_signals"] = [r["predicted_signals"] for r in pred_results]
df_pred["predicted_signals_ko"] = [r["predicted_signals_ko"] for r in pred_results]
df_pred["top3_signals"] = [json.dumps(r["top3_signals"], ensure_ascii=False) for r in pred_results]
df_pred["max_confidence"] = [r["max_confidence"] for r in pred_results]
df_pred["num_predicted_labels"] = [r["num_predicted_labels"] for r in pred_results]
df_pred["needs_human_review"] = [r["needs_human_review"] for r in pred_results]

pred_path = OUTPUT_DIR / "real_reviews_model_predictions.csv"
df_pred.to_csv(pred_path, index=False, encoding="utf-8-sig")

print("예측 결과 저장:", pred_path)
print("분석 리뷰 수:", len(df_pred))
display(df_pred[["review_text", "predicted_signals_ko", "max_confidence", "needs_human_review"]].head(20))


예측 결과 저장: /workspace/outputs/real_reviews_model_predictions.csv
분석 리뷰 수: 1000


,review_text,predicted_signals_ko,max_confidence,needs_human_review
0,압구정로데오카페 부캐드뺑 내돈내산 솔직후기 상하이모찌... 솔직한 후기를 한번 남겨...,"[가격 저항, 공간/분위기 강점]",0.469818,True
1,메종드구르메 우베라떼 압구정로데오 카페 추천 후기 메종드구르메 우베라떼 압구정로데오...,"[공간/분위기 강점, 재방문 의도]",0.588561,False
2,압구정 신상카페 안목 갤러리 나팔 NAFAL 경매 첫 방문 후기 압구정신상카페 안목...,[공간/분위기 강점],0.512684,True
3,추천 압구정 이웃집통통이 버터떡 압구정로데오 카페 후기 메뉴판 압구정 카페 메뉴판인...,"[제품/메뉴 강점, 서비스 리스크]",0.524052,True
4,압구정 베이커리 카페 서울 3대 크루아상 플링크 솔직후기 히히 압구정 베이커리 카페...,[공간/좌석 리스크],0.478938,True
5,"압구정 블루보틀 카페 후기, 아이스 라떼 맛 깔끔한 인테리어가 항상 좋은 블루보틀 ...","[제품/메뉴 강점, 공간/분위기 강점]",0.498446,True
6,압구정카페추천 백억커피 신메뉴 후기 압구정카페추천 백억커피 신메뉴 후기 어느덧 압구...,"[공간/분위기 강점, 재방문 의도]",0.460855,True
7,"신사동] 압구정 카페 부케 드 뺑. 상하이모찌, 소금빵 솔직후기... 저도 후기 찾...","[제품/메뉴 강점, 공간/분위기 강점]",0.471264,True
8,압구정로데오카페 유명한 우베라떼 즐긴 후기 오랜만에 만나는 친구와 함께 수다도 떨겸...,"[공간/분위기 강점, 재방문 의도]",0.608097,False
9,브런치 베이커리 로와이드 LOWIDE 압구정 대형카페 후기 서울 압구정역 브런치 베...,"[제품/메뉴 리스크, 공간/분위기 강점]",0.458634,True


In [41]:

# ============================================================
# 25-5. 운영 신호 요약 + Store DNA 리포트 생성
# ============================================================

signal_counter = Counter()
for labels in df_pred["predicted_signals"]:
    if isinstance(labels, list):
        signal_counter.update(labels)

signal_summary_df = pd.DataFrame([
    {
        "label": label,
        "label_ko": LABEL_KO.get(label, label),
        "count": signal_counter.get(label, 0),
        "ratio": signal_counter.get(label, 0) / max(len(df_pred), 1),
    }
    for label in OPERATION_LABELS
]).sort_values("count", ascending=False)

signal_summary_path = OUTPUT_DIR / "real_reviews_signal_summary.csv"
signal_summary_df.to_csv(signal_summary_path, index=False, encoding="utf-8-sig")

display(signal_summary_df)

STORE_DNA_RULES = {
    "제품 경쟁력": {"positive": ["product_strength"], "negative": ["product_risk"]},
    "서비스 안정성": {"positive": ["service_strength"], "negative": ["service_risk"]},
    "가격 수용성": {"positive": ["price_value_strength"], "negative": ["price_resistance"]},
    "대기/운영 안정성": {"positive": ["operation_efficiency"], "negative": ["waiting_risk"]},
    "공간 매력도": {"positive": ["space_strength"], "negative": ["space_risk"]},
    "청결 안정성": {"positive": ["cleanliness_strength"], "negative": ["cleanliness_risk"]},
    "재방문 신호": {"positive": ["revisit_intent"], "negative": ["revisit_risk"]},
}

def pick_evidence_reviews(df, labels, max_items=3):
    rows = []
    for _, row in df.iterrows():
        pred_labels = row["predicted_signals"]
        if isinstance(pred_labels, list) and any(label in pred_labels for label in labels):
            rows.append(row["review_text"])
        if len(rows) >= max_items:
            break
    return rows

def confidence_from_count(related_count, total):
    ratio = related_count / max(total, 1)
    if related_count >= 50 and ratio >= 0.05:
        return "high"
    if related_count >= 15:
        return "medium"
    return "low"

def build_store_dna_report(df):
    total = max(len(df), 1)
    report = []

    for dimension, rule in STORE_DNA_RULES.items():
        pos_labels = rule["positive"]
        neg_labels = rule["negative"]

        pos_count = sum(signal_counter.get(label, 0) for label in pos_labels)
        neg_count = sum(signal_counter.get(label, 0) for label in neg_labels)
        related_count = pos_count + neg_count

        # 기본 50점에서 긍정/부정 비율 반영
        score = 50 + 50 * ((pos_count - neg_count) / total)
        score = round(max(0, min(100, score)), 1)

        if score >= 70:
            owner_message = f"{dimension}은 현재 강점으로 활용할 수 있습니다."
        elif score >= 45:
            owner_message = f"{dimension}은 보통 수준이며, 리뷰 근거를 추가 확인하는 것이 좋습니다."
        else:
            owner_message = f"{dimension}은 우선 점검이 필요한 신호가 감지됩니다."

        report.append({
            "dimension": dimension,
            "score": score,
            "confidence": confidence_from_count(related_count, total),
            "positive_count": int(pos_count),
            "negative_count": int(neg_count),
            "related_count": int(related_count),
            "evidence_reviews": pick_evidence_reviews(df, pos_labels + neg_labels, max_items=3),
            "owner_message": owner_message,
        })

    return sorted(report, key=lambda x: x["score"], reverse=True)

store_dna_report = build_store_dna_report(df_pred)

store_dna_path = OUTPUT_DIR / "real_reviews_store_dna_report.json"
with open(store_dna_path, "w", encoding="utf-8") as f:
    json.dump(store_dna_report, f, ensure_ascii=False, indent=2)

store_dna_df = pd.DataFrame(store_dna_report)
display(store_dna_df[["dimension", "score", "confidence", "positive_count", "negative_count", "owner_message"]])

print("신호 요약 저장:", signal_summary_path)
print("Store DNA 저장:", store_dna_path)


,label,label_ko,count,ratio
7,space_strength,공간/분위기 강점,569,0.569
0,product_strength,제품/메뉴 강점,394,0.394
11,revisit_intent,재방문 의도,354,0.354
1,product_risk,제품/메뉴 리스크,140,0.140
9,cleanliness_strength,청결 강점,68,0.068
2,service_strength,서비스 강점,67,0.067
12,revisit_risk,재방문 저하,46,0.046
4,price_resistance,가격 저항,24,0.024
3,service_risk,서비스 리스크,21,0.021
5,price_value_strength,가격 대비 만족,16,0.016


,dimension,score,confidence,positive_count,negative_count,owner_message
0,공간 매력도,78.2,high,569,6,공간 매력도은 현재 강점으로 활용할 수 있습니다.
1,재방문 신호,65.4,high,354,46,"재방문 신호은 보통 수준이며, 리뷰 근거를 추가 확인하는 것이 좋습니다."
2,제품 경쟁력,62.7,high,394,140,"제품 경쟁력은 보통 수준이며, 리뷰 근거를 추가 확인하는 것이 좋습니다."
3,청결 안정성,53.4,high,68,0,"청결 안정성은 보통 수준이며, 리뷰 근거를 추가 확인하는 것이 좋습니다."
4,서비스 안정성,52.3,high,67,21,"서비스 안정성은 보통 수준이며, 리뷰 근거를 추가 확인하는 것이 좋습니다."
5,가격 수용성,49.6,medium,16,24,"가격 수용성은 보통 수준이며, 리뷰 근거를 추가 확인하는 것이 좋습니다."
6,대기/운영 안정성,49.5,low,1,12,"대기/운영 안정성은 보통 수준이며, 리뷰 근거를 추가 확인하는 것이 좋습니다."


신호 요약 저장: /workspace/outputs/real_reviews_signal_summary.csv
Store DNA 저장: /workspace/outputs/real_reviews_store_dna_report.json


In [42]:

# ============================================================
# 25-6. 사장님용 요약 리포트 + 내일 할 일 3개
# ============================================================

def make_owner_report(store_dna_report, signal_summary_df, total_reviews):
    sorted_dna = sorted(store_dna_report, key=lambda x: x["score"], reverse=True)
    strengths = sorted_dna[:2]
    risks = sorted_dna[-2:]

    top_signals = signal_summary_df.head(3)

    strength_txt = ", ".join([f"{x['dimension']}({x['score']}점)" for x in strengths])
    risk_txt = ", ".join([f"{x['dimension']}({x['score']}점)" for x in risks])
    top_signal_txt = ", ".join(top_signals["label_ko"].tolist())

    actions = []

    low_dims = [x["dimension"] for x in risks]
    top_labels = signal_summary_df.head(6)["label"].tolist()

    if "price_resistance" in top_labels or "가격 수용성" in low_dims:
        actions.append("가격 저항 신호가 있는 메뉴는 단품 가격 인하보다 세트 구성과 가치 설명 문구를 먼저 테스트하세요.")
    if "waiting_risk" in top_labels or "대기/운영 안정성" in low_dims:
        actions.append("대기/혼잡 신호가 보이면 피크타임 인기 메뉴 3개를 선노출하고 주문 동선을 단순화하세요.")
    if "service_risk" in top_labels or "서비스 안정성" in low_dims:
        actions.append("서비스 리스크 리뷰는 시간대별로 묶어 보고, 바쁜 시간대 응대 멘트와 역할 분담을 먼저 점검하세요.")
    if "product_strength" in top_labels:
        actions.append("제품/메뉴 강점 리뷰 문장은 대표 메뉴 홍보 문구와 매장 소개에 재활용하세요.")
    if "revisit_intent" in top_labels:
        actions.append("재방문 의도 신호가 있는 리뷰는 쿠폰/스탬프/단골 전환 메시지와 연결하세요.")

    # 최소 3개 보장
    default_actions = [
        "POS 데이터의 시간대별 매출과 리뷰 신호를 연결해 실제 병목 시간대를 확인하세요.",
        "리뷰 근거 문장 10개를 직접 읽고 모델 라벨이 맞는지 검수하세요.",
        "상위 강점 1개와 하위 리스크 1개만 골라 이번 주 개선 실험을 설계하세요.",
    ]
    for action in default_actions:
        if len(actions) >= 3:
            break
        if action not in actions:
            actions.append(action)

    report = f"""
# 카페 리뷰 운영 신호 분석 리포트

분석 리뷰 수: {total_reviews:,}건

## 핵심 요약
이번 리뷰 데이터에서 가장 많이 감지된 운영 신호는 **{top_signal_txt}**입니다.

Store DNA 기준 강점은 **{strength_txt}**로 나타났습니다.  
반대로 우선 점검해야 할 항목은 **{risk_txt}**입니다.

## 사장님께 드리는 해석
이 분석은 리뷰를 단순 긍정/부정으로 나누는 것이 아니라, 손님의 말 속에서 매장 운영에 필요한 신호를 뽑아낸 결과입니다.  
다음 단계에서는 이 리뷰 신호를 POS의 시간대별 매출, 메뉴별 매출, 객단가, 주문 집중도와 연결해야 실제 운영 병목을 더 정확하게 판단할 수 있습니다.

## 내일 할 일 3개
1. {actions[0]}
2. {actions[1]}
3. {actions[2]}
""".strip()

    return report, actions[:3]

owner_report, recommended_actions = make_owner_report(store_dna_report, signal_summary_df, len(df_pred))

owner_report_path = OUTPUT_DIR / "real_reviews_owner_report.txt"
with open(owner_report_path, "w", encoding="utf-8") as f:
    f.write(owner_report)

print(owner_report)
print("\n저장:", owner_report_path)


# 카페 리뷰 운영 신호 분석 리포트

분석 리뷰 수: 1,000건

## 핵심 요약
이번 리뷰 데이터에서 가장 많이 감지된 운영 신호는 **공간/분위기 강점, 제품/메뉴 강점, 재방문 의도**입니다.

Store DNA 기준 강점은 **공간 매력도(78.2점), 재방문 신호(65.4점)**로 나타났습니다.  
반대로 우선 점검해야 할 항목은 **가격 수용성(49.6점), 대기/운영 안정성(49.5점)**입니다.

## 사장님께 드리는 해석
이 분석은 리뷰를 단순 긍정/부정으로 나누는 것이 아니라, 손님의 말 속에서 매장 운영에 필요한 신호를 뽑아낸 결과입니다.  
다음 단계에서는 이 리뷰 신호를 POS의 시간대별 매출, 메뉴별 매출, 객단가, 주문 집중도와 연결해야 실제 운영 병목을 더 정확하게 판단할 수 있습니다.

## 내일 할 일 3개
1. 가격 저항 신호가 있는 메뉴는 단품 가격 인하보다 세트 구성과 가치 설명 문구를 먼저 테스트하세요.
2. 대기/혼잡 신호가 보이면 피크타임 인기 메뉴 3개를 선노출하고 주문 동선을 단순화하세요.
3. 제품/메뉴 강점 리뷰 문장은 대표 메뉴 홍보 문구와 매장 소개에 재활용하세요.

저장: /workspace/outputs/real_reviews_owner_report.txt



# 26. Codex 서비스화 연결

이 노트북에서 서비스 코드로 옮겨야 할 핵심 기능은 아래 5개입니다.

```text
1. 모델 로드
   - AutoTokenizer.from_pretrained(MODEL_DIR)
   - AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

2. CSV 업로드/전처리
   - title + snippet → review_text
   - 중복/공백 제거

3. 다중 라벨 예측
   - sigmoid
   - label_threshold 적용
   - top3 signal 생성

4. Store DNA 변환
   - product/service/price/waiting/space/cleanliness/revisit 차원 점수화
   - confidence와 근거 리뷰 추출

5. 사장님용 리포트
   - 핵심 요약
   - 강점/리스크
   - 내일 할 일 3개
```

FastAPI 기준 첫 MVP 엔드포인트는 다음 3개면 충분합니다.

```text
GET  /health
POST /analyze-reviews
GET  /download/{job_id}/{file_type}
```

이제부터 Codex와 할 일은 **새 모델을 만드는 것**이 아니라,  
이 노트북의 후반부 추론 파이프라인을 `backend/app/services/` 코드로 분리하는 것입니다.
